In [1]:
!pip install datasets anndata scanpy scipy pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 67.8 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
import os

# Paste your HF token here, or set the HF_TOKEN environment variable
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()  # prompts interactive login widget in Colab / Jupyter


In [2]:
!curl -L -A "Mozilla/5.0" -o K562.h5ad \
"https://ndownloader.figshare.com/files/35773219"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  9.9G  100  9.9G    0     0  15.6M      0  0:10:50  0:10:50 --:--:-- 16.1M


In [ ]:
# --- DIAGNOSTIC 2 --- find control label
import scanpy as sc
adata = sc.read_h5ad("/content/K562.h5ad")

vals = adata.obs["gene"].value_counts()
print("Top 10 most frequent gene labels (control is usually the largest):")
print(vals.head(10))
print(f"\nTotal unique conditions: {adata.obs['gene'].nunique()}")

Top 10 most frequent gene labels (control is usually the largest):
gene
non-targeting    10691
RPL3              1996
NCBP2              992
KIF11              974
SLC39A9            752
DONSON             745
NUP205             735
GAB2               674
BUB1               648
SNX15              637
Name: count, dtype: int64

Total unique conditions: 2058


In [ ]:
# ============================================================
# GEARS EVALUATION — Zero-shot transfer: Norman → K562
# Uses pretrained Norman model, evaluates on K562 ground truth
# on overlapping gene vocabulary
# ============================================================

# ─── 0. INSTALL ──────────────────────────────────────────────
import subprocess, sys, torch

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--break-system-packages", *pkgs])

# torch_geometric must come BEFORE cell-gears
try:
    import torch_geometric
    print("torch_geometric : already installed")
except ImportError:
    print("Installing torch_geometric ...", flush=True)
    pip("torch_geometric")
    print("torch_geometric : done")

try:
    import torch_scatter
    print("torch_scatter   : already installed")
except ImportError:
    _tv  = torch.__version__.split("+")[0]
    _cud = torch.version.cuda.replace(".", "")
    try:
        pip("torch_scatter", "torch_sparse",
            "-f", f"https://data.pyg.org/whl/torch-{_tv}+cu{_cud}.html")
        print("torch_scatter   : done")
    except Exception as _e:
        print(f"torch_scatter   : skipped ({_e})")

pip("cell-gears", "scanpy")
print("cell-gears / scanpy : done")

# Monkey-patch pandas 2.0 (Series.nonzero removed)
import pandas as pd
if not hasattr(pd.Series, "nonzero"):
    pd.Series.nonzero = lambda self: self.to_numpy().nonzero()

# ─── 1. IMPORTS ──────────────────────────────────────────────
import os, warnings, time
import numpy as np
import scanpy as sc
import scipy.sparse as sp
from zipfile import ZipFile
from gears import PertData, GEARS
from gears.utils import dataverse_download

warnings.filterwarnings("ignore")

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
D          = torch.device(DEVICE)
MODEL_NAME = "GEARS (Norman→K562 zero-shot)"
TOP_K_DE   = 50
SEED       = 1
CTRL_LABEL = "non-targeting"

print(f"\nDevice : {DEVICE}")
print(f"numpy  : {np.__version__}")
print(f"torch  : {torch.__version__}")

# ─── 2. DOWNLOAD Norman data + pretrained model ───────────────
NORMAN_DATA_PATH  = "./norman_umi_go"
NORMAN_MODEL_CKPT = "./model_ckpt"

if not os.path.exists(NORMAN_DATA_PATH):
    print("\n[1/5] Downloading Norman data ...", flush=True)
    dataverse_download(
        'https://dataverse.harvard.edu/api/access/datafile/6979957',
        'norman_umi_go.tar.gz'
    )
    import tarfile
    with tarfile.open('norman_umi_go.tar.gz', 'r:gz') as tar:
        tar.extractall()
    print("  Norman data extracted.")
else:
    print(f"\n[1/5] Norman data : found locally ({NORMAN_DATA_PATH})")

if not os.path.exists(NORMAN_MODEL_CKPT):
    print("[2/5] Downloading pretrained GEARS model ...", flush=True)
    dataverse_download(
        'https://dataverse.harvard.edu/api/access/datafile/10457098',
        'model.zip'
    )
    with ZipFile('model.zip', 'r') as z:
        z.extractall(path='./')
    print("  Model extracted.")
else:
    print(f"[2/5] GEARS model : found locally ({NORMAN_MODEL_CKPT})")

# ─── 3. LOAD Norman PertData + model ─────────────────────────
print("\n[3/5] Loading Norman PertData + model ...", flush=True)
pert_data = PertData('./')
pert_data.load(data_path=NORMAN_DATA_PATH)
pert_data.prepare_split(split='no_test', seed=SEED)
pert_data.get_dataloader(batch_size=32, test_batch_size=128)

gears_model = GEARS(pert_data, device=DEVICE,
                    weight_bias_track=False,
                    proj_name='gears',
                    exp_name='gears_misc_umi_no_test')
gears_model.model_initialize(hidden_size=64)
gears_model.load_pretrained(NORMAN_MODEL_CKPT)

norman_genes     = set(pert_data.gene_names.tolist())
norman_gene_list = pert_data.gene_names.tolist()
print(f"  Norman vocab : {len(norman_genes)} genes")

# ─── 4. LOAD K562 ground truth ───────────────────────────────
print("\n[4/5] Loading K562 h5ad ...", flush=True)
adata = sc.read_h5ad("/content/K562.h5ad")
print(f"  Loaded : {adata.shape}")

adata.obs["condition"] = adata.obs["gene"].apply(
    lambda g: "ctrl" if g == CTRL_LABEL else f"{g}+ctrl"
)
if "gene_name" not in adata.var.columns:
    adata.var["gene_name"] = adata.var.index

# 20% stratified subsample (same fraction used across all models)
np.random.seed(SEED)
keep_idx = []
for cond, grp in adata.obs.groupby("condition"):
    n = max(1, int(len(grp) * 0.2))
    keep_idx.extend(np.random.choice(grp.index, size=n, replace=False).tolist())
adata = adata[keep_idx].copy()
print(f"  Subsampled : {adata.shape}  (20% stratified)")

counts = adata.obs["condition"].value_counts()
adata  = adata[adata.obs["condition"].isin(counts[counts >= 5].index)].copy()
print(f"  Min-5 filter : {adata.shape} | {adata.obs['condition'].nunique()-1} perts")

if not sp.issparse(adata.X):
    adata.X = sp.csr_matrix(adata.X)

k562_genes   = adata.var["gene_name"].tolist()
X_ctrl_raw   = adata[adata.obs["condition"] == "ctrl"].X.toarray()
# Log-normalize to match GEARS prediction space (log1p CPM)
_lib         = X_ctrl_raw.sum(axis=1, keepdims=True)
X_ctrl_ln    = np.log1p(X_ctrl_raw / (_lib + 1e-8) * 1e4)
ctrl_mean_k562 = X_ctrl_ln.mean(axis=0)   # (n_k562_genes,)

# ─── 5. GENE & PERTURBATION OVERLAP ──────────────────────────
overlap_genes  = norman_genes & set(k562_genes)
overlap_sorted = sorted(overlap_genes)
norman_idx     = np.array([norman_gene_list.index(g) for g in overlap_sorted])
k562_idx       = np.array([k562_genes.index(g) for g in overlap_sorted])

ctrl_mean_overlap = torch.tensor(
    ctrl_mean_k562[k562_idx], dtype=torch.float32
).to(D)

k562_perts = sorted([
    c for c in adata.obs["condition"].unique()
    if c != "ctrl" and c.replace("+ctrl", "") in norman_genes
])

print(f"\n  Gene overlap : Norman={len(norman_genes)}, "
      f"K562={len(k562_genes)}, Overlap={len(overlap_genes)}")
print(f"  Predictable perts : {len(k562_perts)}")

# ─── 6. ZERO-SHOT INFERENCE ──────────────────────────────────
print(f"\n[5/5] Running zero-shot predictions ({len(k562_perts)} perts) ...",
      flush=True)

pred_list, true_list, pert_names = [], [], []
true_cells_dict, pred_point_dict = {}, {}
n_skip = 0
t0 = time.time()

for i, pert in enumerate(k562_perts):
    gene = pert.replace("+ctrl", "")
    mask = adata.obs["condition"] == pert
    if mask.sum() < 2:
        n_skip += 1
        continue

    # Ground truth — log-normalize to match GEARS prediction space
    X_true_raw        = adata[mask].X.toarray()
    _lib              = X_true_raw.sum(axis=1, keepdims=True)
    X_true            = np.log1p(X_true_raw / (_lib + 1e-8) * 1e4)
    true_mean_overlap = X_true.mean(axis=0)[k562_idx]

    # GEARS prediction
    try:
        pred_dict    = gears_model.predict([[gene]])
        pred_full    = np.array(list(pred_dict.values())[0])
        pred_overlap = pred_full[norman_idx]
    except Exception:
        n_skip += 1
        continue

    pred_list.append(torch.tensor(pred_overlap,       dtype=torch.float32).to(D))
    true_list.append(torch.tensor(true_mean_overlap,  dtype=torch.float32).to(D))
    true_cells_dict[pert] = torch.tensor(
        X_true[:, k562_idx], dtype=torch.float32).to(D)
    pred_point_dict[pert] = torch.tensor(
        pred_overlap, dtype=torch.float32).unsqueeze(0).to(D)
    pert_names.append(pert)

    # Progress every 50 perts
    if (i + 1) % 50 == 0 or (i + 1) == len(k562_perts):
        elapsed = time.time() - t0
        rate    = (i + 1) / elapsed
        eta     = (len(k562_perts) - i - 1) / rate
        print(f"  {i+1:>4}/{len(k562_perts)}  done={len(pert_names)}  "
              f"skip={n_skip}  {elapsed:.0f}s elapsed  ETA {eta:.0f}s",
              flush=True)

pred_centroids = torch.stack(pred_list)   # (P, n_overlap)
true_centroids = torch.stack(true_list)
pred_delta     = pred_centroids - ctrl_mean_overlap.unsqueeze(0)
true_delta     = true_centroids - ctrl_mean_overlap.unsqueeze(0)

print(f"\n  Collected : {len(pert_names)} perts | shape {pred_centroids.shape}")
print(f"  Skipped   : {n_skip}")

# ─── 7. METRICS ──────────────────────────────────────────────
print("\nComputing metrics ...", flush=True)

# ── T1 ────────────────────────────────────────────────────────
def systema_metrics(pred_c, true_c):
    n     = pred_c.size(0)
    dists = torch.cdist(pred_c, true_c, p=2.0)
    d_t   = dists.diag()
    ca    = (d_t.unsqueeze(1) < dists).sum(dim=1).float() / (n - 1)
    mask  = ~torch.eye(n, dtype=torch.bool, device=pred_c.device)
    pds   = d_t / (d_t + (dists * mask).sum(dim=1) / (n - 1) + 1e-8)
    return ca.mean().item(), pds.mean().item()

def systema_pearson_delta(pred_c, true_c):
    o  = true_c.mean(dim=0, keepdim=True)
    pc = pred_c - o;  tc = true_c - o
    pc = pc - pc.mean(-1, keepdim=True)
    tc = tc - tc.mean(-1, keepdim=True)
    cov = (pc * tc).sum(-1)
    return (cov / ((pc**2).sum(-1).clamp(1e-8).sqrt() *
                   (tc**2).sum(-1).clamp(1e-8).sqrt())).mean().item()

# ── T2 ────────────────────────────────────────────────────────
def directional_accuracy(pred_d, true_d, thr=0.1):
    active  = true_d.abs() > thr
    matches = (torch.sign(pred_d) == torch.sign(true_d)) & active
    return (matches.sum(-1).float() /
            (active.sum(-1).float() + 1e-8)).mean().item()

def pearson_delta_topk(pred_d, true_d, k=TOP_K_DE):
    k   = min(k, pred_d.size(1))
    _, idx = torch.topk(true_d.abs(), k=k, dim=-1)
    pt  = torch.gather(pred_d, 1, idx); tt = torch.gather(true_d, 1, idx)
    pc  = pt - pt.mean(-1, keepdim=True); tc = tt - tt.mean(-1, keepdim=True)
    cov = (pc * tc).sum(-1)
    return (cov / ((pc**2).sum(-1).clamp(1e-8).sqrt() *
                   (tc**2).sum(-1).clamp(1e-8).sqrt())).mean().item()

def de_gene_jaccard(pred_d, true_d, k=TOP_K_DE):
    k = min(k, pred_d.size(1))
    _, ti = torch.topk(true_d.abs(), k=k, dim=-1)
    _, pi = torch.topk(pred_d.abs(), k=k, dim=-1)
    return float(np.mean([
        len(set(t.tolist()) & set(p.tolist())) /
        len(set(t.tolist()) | set(p.tolist()))
        for t, p in zip(ti, pi)
    ]))

# ── T3 — point mass (GEARS gives pseudobulk mean) vs true cells ──
# Both pred and true are now in log-norm CPM space → comparable distances.
def energy_distance_point(pred_pt, x_true, max_cells=512):
    if x_true.size(0) > max_cells:
        idx    = torch.randperm(x_true.size(0), device=x_true.device)[:max_cells]
        x_true = x_true[idx]
    d_pt = torch.cdist(pred_pt, x_true, p=2.0).mean()
    d_tt = torch.cdist(x_true, x_true, p=2.0).mean()
    return max((2 * d_pt - d_tt).item(), 0.0)

def mmd_rbf_point(pred_pt, x_true, max_cells=512):
    """MMD with RBF kernel (median bandwidth heuristic) — distinct from energy distance."""
    if x_true.size(0) > max_cells:
        idx    = torch.randperm(x_true.size(0), device=x_true.device)[:max_cells]
        x_true = x_true[idx]
    with torch.no_grad():
        sigma = torch.cdist(x_true, x_true).median().clamp(1e-3).item()
    def rbf(a, b): return torch.exp(-torch.cdist(a, b).pow(2) / (2 * sigma ** 2))
    kxx = rbf(pred_pt, pred_pt).mean()
    kyy = rbf(x_true, x_true).mean()
    kxy = rbf(pred_pt, x_true).mean()
    return max((kxx - 2 * kxy + kyy).item(), 0.0)

# ── Compute ────────────────────────────────────────────────────
ca, pds = systema_metrics(pred_centroids, true_centroids)
print(f"  T1 done", flush=True)

d_sys   = systema_pearson_delta(pred_centroids, true_centroids)
dir_acc = directional_accuracy(pred_delta, true_delta)
pear_de = pearson_delta_topk(pred_delta, true_delta)
jaccard = de_gene_jaccard(pred_delta, true_delta)
print(f"  T2 done", flush=True)

e_scores, mmd_scores = [], []
n_perts = len(pert_names)
t0_t3   = time.time()
for i, pert in enumerate(pert_names):
    e_scores.append(energy_distance_point(pred_point_dict[pert],
                                          true_cells_dict[pert]))
    mmd_scores.append(mmd_rbf_point(pred_point_dict[pert],
                                    true_cells_dict[pert]))
    if (i + 1) % 50 == 0 or (i + 1) == n_perts:
        elapsed = time.time() - t0_t3
        eta     = elapsed / (i + 1) * (n_perts - i - 1)
        print(f"  T3 {i+1:>4}/{n_perts}  {elapsed:.0f}s elapsed  ETA {eta:.0f}s",
              flush=True)

e_dist  = float(np.mean(e_scores))
mmd_val = float(np.mean(mmd_scores))

# ─── 8. DISPLAY ──────────────────────────────────────────────
SEP = "=" * 62
print(f"\n{SEP}")
print(f"  {MODEL_NAME}")
print(f"  K562 Perturb-seq | 20% subsample | {len(overlap_sorted)} overlap genes")
print(f"  {len(pert_names)} perturbations evaluated")
print(f"  {'Tier Metric':<40} {'Value':>8}  Dir")
print(f"  {'-'*56}")

def _row(tag, name, val, hi):
    print(f"  [{tag}] {name:<38} {val:>8.4f}  {'↑' if hi else '↓'}")

_row("T1", "Centroid Accuracy (CA)",       ca,      hi=True)
_row("T1", "Profile Distance Score (PDS)", pds,     hi=False)
_row("T1", "Systema Pearson Δ",            d_sys,   hi=True)
_row("T2", "Directional Accuracy",         dir_acc, hi=True)
_row("T2", f"Pearson Δ DE (Top-{TOP_K_DE})", pear_de, hi=True)
_row("T2", f"Jaccard DE (Top-{TOP_K_DE})",   jaccard, hi=True)
_row("T3", "Energy Distance",              e_dist,  hi=False)
_row("T3", "MMD (RBF kernel)",             mmd_val, hi=False)
print(SEP)
print("Done.")

Installing torch_geometric ...
torch_geometric : done
torch_scatter   : done
cell-gears / scanpy : done

Device : cuda
numpy  : 2.0.2
torch  : 2.10.0+cu128

[1/5] Downloading Norman data ...


Downloading...
100%|██████████| 1.10G/1.10G [00:27<00:00, 39.5MiB/s]


  Norman data extracted.
[2/5] Downloading pretrained GEARS model ...


Downloading...
100%|██████████| 8.50M/8.50M [00:00<00:00, 10.9MiB/s]


  Model extracted.

[3/5] Loading Norman PertData + model ...


Downloading...
100%|██████████| 9.46M/9.46M [00:00<00:00, 11.0MiB/s]
Downloading...
100%|██████████| 559k/559k [00:00<00:00, 1.61MiB/s]
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!
Creating new splits....
Saving new splits at ./norman_umi_go/splits/norman_umi_go_no_test_1_0.75.pkl
Done!
Creating dataloaders....
Done!
Downloading...
100%|██████████| 60.7M/60.7M [00:02<00:00, 28.8MiB/s]
Extracting tar file...
Done!


  Norman vocab : 5054 genes

[4/5] Loading K562 h5ad ...
  Loaded : (310385, 8563)
  Subsampled : (61244, 8563)  (20% stratified)
  Min-5 filter : (61093, 8563) | 2003 perts

  Gene overlap : Norman=5054, K562=8563, Overlap=2961
  Predictable perts : 1003

[5/5] Running zero-shot predictions (1003 perts) ...
    50/1003  done=50  skip=0  8s elapsed  ETA 158s
   100/1003  done=99  skip=1  15s elapsed  ETA 136s
   150/1003  done=149  skip=1  23s elapsed  ETA 133s
   200/1003  done=199  skip=1  30s elapsed  ETA 122s
   250/1003  done=249  skip=1  37s elapsed  ETA 112s
   300/1003  done=297  skip=3  44s elapsed  ETA 103s
   350/1003  done=347  skip=3  51s elapsed  ETA 95s
   400/1003  done=397  skip=3  58s elapsed  ETA 87s
   450/1003  done=447  skip=3  65s elapsed  ETA 80s
   500/1003  done=497  skip=3  72s elapsed  ETA 72s
   550/1003  done=547  skip=3  80s elapsed  ETA 66s
   600/1003  done=597  skip=3  87s elapsed  ETA 59s
   650/1003  done=647  skip=3  94s elapsed  ETA 51s
   700/1003

In [ ]:
import subprocess, sys
print("🔧 Reinstalling SciPy to repair the broken .so extension...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--force-reinstall", "--no-deps", "scipy"
])
print("✅ SciPy reinstalled (same version, just repaired).")


🔧 Reinstalling SciPy to repair the broken .so extension...
✅ SciPy reinstalled (same version, just repaired).


# restart session here


In [ ]:
# ============================================================
# STATE EVALUATION — K562 Perturb-seq | Full Metrics Suite
# Model  : arcinstitute/ST-HVG-Replogle (HuggingFace)
# Mode   : zero-shot inference — no K562 training data used
# Pipeline:
#   1. preprocess_train : raw counts → log-norm → 2000 HVGs → obsm["X_hvg"]
#   2. infer            : obsm["X_hvg"] → per-cell predictions → pred_adata.obsm["X_hvg"]
# ============================================================

import subprocess, sys, os, glob, shutil, warnings, time
import numpy as np
import torch

# ─── 0. INSTALL ──────────────────────────────────────────────
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--break-system-packages", *args])

pip("uv", "huggingface_hub")

_UV_BIN = os.path.expanduser("~/.local/bin")
if _UV_BIN not in os.environ.get("PATH", ""):
    os.environ["PATH"] = _UV_BIN + ":" + os.environ.get("PATH", "")

if shutil.which("state") is None:
    print("Installing arc-state via uv tool install ...", flush=True)
    subprocess.check_call(["uv", "tool", "install", "arc-state"])
    print("arc-state installed.")
else:
    print(f"arc-state : {shutil.which('state')}")

# pip above may upgrade scipy on disk; evict stale module cache so the
# reimport picks up the newly installed .so rather than the old cached one.
# DO NOT remove — without this, scipy raises ImportError on dia_tocsr.
for _k in list(sys.modules.keys()):
    if any(_k == m or _k.startswith(m + ".")
           for m in ("scanpy", "anndata", "scipy", "pandas")):
        del sys.modules[_k]

# ─── 1. IMPORTS ──────────────────────────────────────────────
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from huggingface_hub import snapshot_download

warnings.filterwarnings("ignore")

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
CTRL_LABEL = "non-targeting"
TOP_K_DE   = 50
SEED       = 42
HF_REPO    = "arcinstitute/ST-HVG-Replogle"
LOCAL_DIR  = "ST-HVG-Replogle"
MODEL_NAME = "STATE (ST-HVG-Replogle, zero-shot)"

print(f"Device : {DEVICE}")
print(f"numpy  : {np.__version__}")
print(f"torch  : {torch.__version__}")

# ─── 2. DOWNLOAD MODEL ───────────────────────────────────────
if not os.path.exists(LOCAL_DIR):
    print(f"\n[1/6] Downloading {HF_REPO} ...", flush=True)
    t0 = time.time()
    snapshot_download(repo_id=HF_REPO, local_dir=LOCAL_DIR)
    print(f"  Done in {time.time()-t0:.0f}s")
else:
    print(f"\n[1/6] Model : found locally ({LOCAL_DIR}/)")

_ckpts = sorted(glob.glob(f"{LOCAL_DIR}/**/best.ckpt", recursive=True))
if not _ckpts:
    _ckpts = sorted(glob.glob(f"{LOCAL_DIR}/**/*.ckpt", recursive=True))
if not _ckpts:
    raise FileNotFoundError(
        f"No .ckpt found under '{LOCAL_DIR}'. "
        "Check that snapshot_download completed successfully."
    )
CHECKPOINT = _ckpts[0]
MODEL_DIR  = os.path.dirname(os.path.dirname(CHECKPOINT))
print(f"  Model dir  : {MODEL_DIR}")
print(f"  Checkpoint : {CHECKPOINT}")

# ─── 3. LOAD K562 ────────────────────────────────────────────
print("\n[2/6] Loading K562 h5ad ...", flush=True)
adata = sc.read_h5ad("/content/K562.h5ad")
print(f"  Loaded : {adata.shape}")

if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()
adata.X = adata.layers["counts"].copy()

adata.obs["gene"] = adata.obs["gene"].astype(str)

if "gem_group" not in adata.obs.columns:
    adata.obs["gem_group"] = "batch1"
    print("  Added dummy gem_group='batch1'")

# ─── 4. STRATIFIED 20% SUBSAMPLE ─────────────────────────────
np.random.seed(SEED)
keep_idx = []
for cond, grp in adata.obs.groupby("gene"):
    n = max(2, int(len(grp) * 0.2))
    keep_idx.extend(
        np.random.choice(grp.index, size=n, replace=False).tolist()
    )
adata = adata[keep_idx].copy()

cnt   = adata.obs["gene"].value_counts()
adata = adata[adata.obs["gene"].isin(cnt[cnt >= 5].index)].copy()
print(f"  Subsampled : {adata.shape}  (20% stratified)")
print(f"  Perts      : {adata.obs['gene'].nunique() - 1}")

# ─── 5. WRITE RAW h5ad ───────────────────────────────────────
raw_path  = "K562_raw.h5ad"
pre_path  = "K562_preprocessed.h5ad"
pred_path = "K562_predicted.h5ad"
adata.write_h5ad(raw_path)
print(f"  Wrote raw h5ad → {raw_path}")

# ─── 6. STATE PREPROCESS ─────────────────────────────────────
print("\n[3/6] Preprocessing (norm → log1p → 2000 HVGs) ...", flush=True)
t0 = time.time()
subprocess.check_call([
    "state", "tx", "preprocess_train",
    "--adata",    raw_path,
    "--output",   pre_path,
    "--num_hvgs", "2000",
])
print(f"  Done in {time.time()-t0:.0f}s → {pre_path}")

# ─── 7. STATE INFERENCE ──────────────────────────────────────
print("\n[4/6] Running STATE inference (zero-shot) ...", flush=True)
t0 = time.time()
subprocess.check_call([
    "state", "tx", "infer",
    "--model-dir",    MODEL_DIR,
    "--checkpoint",   CHECKPOINT,
    "--adata",        pre_path,
    "--pert-col",     "gene",
    "--batch-col",    "gem_group",
    "--control-pert", CTRL_LABEL,
    "--embed-key",    "X_hvg",
    "--output",       pred_path,
])
print(f"  Done in {time.time()-t0:.0f}s → {pred_path}")

# ─── 8. LOAD & VALIDATE RESULTS ──────────────────────────────
print("\n[5/6] Loading predictions ...", flush=True)
true_adata = sc.read_h5ad(pre_path)
pred_adata = sc.read_h5ad(pred_path)

assert len(pred_adata) == len(true_adata), \
    f"Row mismatch: pred {len(pred_adata)} vs true {len(true_adata)}"

# STATE infer output layout:
#   pred_adata.X             = input log-norm (passed through — NOT predictions)
#   pred_adata.obsm["X_hvg"] = actual model predictions in 2000-HVG space
#   true_adata.obsm["X_hvg"] = observed log-norm HVG expression (ground truth)
true_X     = np.array(true_adata.obsm["X_hvg"], dtype=np.float32)   # (N, 2000)
pred_X     = np.array(pred_adata.obsm["X_hvg"], dtype=np.float32)   # (N, 2000)
conditions = pred_adata.obs["gene"].values

assert true_X.shape == pred_X.shape, \
    f"Shape mismatch: true {true_X.shape} vs pred {pred_X.shape}"

print(f"  true_X : {true_X.shape}  [{true_X.min():.3f}, {true_X.max():.3f}]")
print(f"  pred_X : {pred_X.shape}  [{pred_X.min():.3f}, {pred_X.max():.3f}]")

# ─── 9. PER-PERTURBATION CENTROIDS ───────────────────────────
ctrl_mask = conditions == CTRL_LABEL
if ctrl_mask.sum() == 0:
    raise ValueError(
        f"No control cells found with CTRL_LABEL='{CTRL_LABEL}'. "
        f"Unique genes: {np.unique(conditions)[:10]}"
    )

ctrl_mu = true_X[ctrl_mask].mean(0)
ctrl_t  = torch.tensor(ctrl_mu, dtype=torch.float32)

pred_list, true_list, pert_names = [], [], []
pred_cells_d, true_cells_d = {}, {}

for pert in np.unique(conditions):
    if pert == CTRL_LABEL:
        continue
    mask = conditions == pert
    if mask.sum() < 2:
        continue
    pred_list.append(torch.tensor(pred_X[mask].mean(0), dtype=torch.float32))
    true_list.append(torch.tensor(true_X[mask].mean(0), dtype=torch.float32))
    pred_cells_d[pert] = torch.tensor(pred_X[mask], dtype=torch.float32)
    true_cells_d[pert] = torch.tensor(true_X[mask], dtype=torch.float32)
    pert_names.append(pert)

pred_c = torch.stack(pred_list)   # (P, G)
true_c = torch.stack(true_list)
pred_d = pred_c - ctrl_t
true_d = true_c - ctrl_t
print(f"  Centroids : {len(pert_names)} perts | shape {pred_c.shape}")

# ─── 10. METRICS ─────────────────────────────────────────────
print("\n[6/6] Computing metrics ...", flush=True)

# ── T1 ────────────────────────────────────────────────────────
def _ca_pds(pc, tc):
    n   = pc.size(0)
    D   = torch.cdist(pc, tc, p=2.0)
    d_t = D.diag()
    off = ~torch.eye(n, dtype=torch.bool, device=D.device)
    ca  = (d_t.unsqueeze(1) < D).float().sum(1) / max(n - 1, 1)
    pds = d_t / (d_t + (D * off).sum(1) / max(n - 1, 1) + 1e-8)
    return ca.mean().item(), pds.mean().item()

def _pearson_delta(pc, tc):
    o = tc.mean(0, keepdim=True)
    p = pc - o;  t = tc - o
    p = p - p.mean(-1, keepdim=True)
    t = t - t.mean(-1, keepdim=True)
    cov = (p * t).sum(-1)
    return (cov / (p.pow(2).sum(-1).clamp(1e-8).sqrt() *
                   t.pow(2).sum(-1).clamp(1e-8).sqrt())).mean().item()

print("  T1 ...", end=" ", flush=True)
ca, pds = _ca_pds(pred_c, true_c)
d_sys   = _pearson_delta(pred_c, true_c)
print("done")

# ── T2 ────────────────────────────────────────────────────────
def _dir_acc(pd_, td_, thr=0.1):
    active  = td_.abs() > thr
    matches = (pd_.sign() == td_.sign()) & active
    return (matches.sum(-1).float() /
            (active.sum(-1).float() + 1e-8)).mean().item()

def _pearson_de(pd_, td_, k=TOP_K_DE):
    k   = min(k, pd_.size(1))
    idx = td_.abs().topk(k, dim=-1).indices
    p   = pd_.gather(1, idx);  t = td_.gather(1, idx)
    p   = p - p.mean(-1, keepdim=True)
    t   = t - t.mean(-1, keepdim=True)
    cov = (p * t).sum(-1)
    return (cov / (p.pow(2).sum(-1).clamp(1e-8).sqrt() *
                   t.pow(2).sum(-1).clamp(1e-8).sqrt())).mean().item()

def _jaccard(pd_, td_, k=TOP_K_DE):
    k  = min(k, pd_.size(1))
    ti = td_.abs().topk(k, dim=-1).indices
    pi = pd_.abs().topk(k, dim=-1).indices
    return float(np.mean([
        len(set(t.tolist()) & set(p.tolist())) /
        len(set(t.tolist()) | set(p.tolist()))
        for t, p in zip(ti, pi)
    ]))

print("  T2 ...", end=" ", flush=True)
dir_acc = _dir_acc(pred_d, true_d)
pear_de = _pearson_de(pred_d, true_d)
jaccard = _jaccard(pred_d, true_d)
print("done")

# ── T3 ────────────────────────────────────────────────────────
def _energy(xp, xt, n=512):
    if xp.size(0) > n: xp = xp[torch.randperm(xp.size(0))[:n]]
    if xt.size(0) > n: xt = xt[torch.randperm(xt.size(0))[:n]]
    return (2 * torch.cdist(xp, xt).mean()
              - torch.cdist(xp, xp).mean()
              - torch.cdist(xt, xt).mean()).item()

def _mmd(xp, xt, n=512):
    if xp.size(0) > n: xp = xp[torch.randperm(xp.size(0))[:n]]
    if xt.size(0) > n: xt = xt[torch.randperm(xt.size(0))[:n]]
    kxx = -torch.cdist(xp, xp).mean()
    kyy = -torch.cdist(xt, xt).mean()
    kxy = -torch.cdist(xp, xt).mean()
    return (kxx - 2 * kxy + kyy).item()

e_scores, mmd_scores = [], []
t0 = time.time()
n_perts = len(pert_names)
for i, pert in enumerate(pert_names):
    e_scores.append(_energy(pred_cells_d[pert], true_cells_d[pert]))
    mmd_scores.append(_mmd(pred_cells_d[pert], true_cells_d[pert]))
    if (i + 1) % 50 == 0 or (i + 1) == n_perts:
        elapsed = time.time() - t0
        eta     = elapsed / (i + 1) * (n_perts - i - 1)
        print(f"  T3 {i+1:>4}/{n_perts}  {elapsed:.0f}s elapsed  ETA {eta:.0f}s",
              flush=True)

e_dist  = float(np.mean(e_scores))
mmd_val = float(np.mean(mmd_scores))

# ─── 11. RESULTS ─────────────────────────────────────────────
_rows = [
    ("T1", "Centroid Accuracy (CA)",        "↑", ca),
    ("T1", "Profile Distance Score (PDS)",  "↓", pds),
    ("T1", "Systema Pearson Δ",             "↑", d_sys),
    ("T2", "Directional Accuracy",          "↑", dir_acc),
    ("T2", f"Pearson Δ DE Top-{TOP_K_DE}",  "↑", pear_de),
    ("T2", f"Jaccard DE Top-{TOP_K_DE}",    "↑", jaccard),
    ("T3", "Energy Distance",               "↓", e_dist),
    ("T3", "MMD (energy kernel)",           "↓", mmd_val),
]

SEP = "=" * 62
print(f"\n{SEP}")
print(f"  {MODEL_NAME}")
print(f"  K562 Perturb-seq | 20% subsample | 2000 HVGs | log-norm space")
print(f"  {len(pert_names)} perturbations evaluated")
print(f"  {'Tier Metric':<40} {'Value':>8}  Dir")
print(f"  {'-'*56}")
for tier, name, d, val in _rows:
    print(f"  [{tier}] {name:<38} {val:>8.4f}  {d}")
print(SEP)
print("Done.")

Installing arc-state via uv tool install ...
arc-state installed.
Device : cuda
numpy  : 2.0.2
torch  : 2.10.0+cu128

[1/6] Downloading arcinstitute/ST-HVG-Replogle ...


Fetching 193 files:   0%|          | 0/193 [00:00<?, ?it/s]

  Done in 165s
  Model dir  : ST-HVG-Replogle/fewshot/hepg2
  Checkpoint : ST-HVG-Replogle/fewshot/hepg2/checkpoints/best.ckpt

[2/6] Loading K562 h5ad ...
  Loaded : (310385, 8563)
  Subsampled : (61093, 8563)  (20% stratified)
  Perts      : 2003
  Wrote raw h5ad → K562_raw.h5ad

[3/6] Preprocessing (norm → log1p → 2000 HVGs) ...
  Done in 40s → K562_preprocessed.h5ad

[4/6] Running STATE inference (zero-shot) ...
  Done in 55s → K562_predicted.h5ad

[5/6] Loading predictions ...
  true_X : (61093, 2000)  [0.000, 8.315]
  pred_X : (61093, 2000)  [0.000, 9.801]
  Centroids : 2003 perts | shape torch.Size([2003, 2000])

[6/6] Computing metrics ...
  T1 ... done
  T2 ... done
  T3   50/2003  0s elapsed  ETA 4s
  T3  100/2003  0s elapsed  ETA 4s
  T3  150/2003  0s elapsed  ETA 4s
  T3  200/2003  0s elapsed  ETA 4s
  T3  250/2003  0s elapsed  ETA 3s
  T3  300/2003  1s elapsed  ETA 3s
  T3  350/2003  1s elapsed  ETA 3s
  T3  400/2003  1s elapsed  ETA 3s
  T3  450/2003  1s elapsed  ETA 3s
 

restart here

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  scGPT · K562 Perturb-seq · Zero-Shot Eval  (h5ad-only, no training)
#
#  Strategy: pre-trained scGPT_human weights only. For each KO gene we feed
#  control cells through the model with that gene's pert_flag=1, collect the
#  predicted post-KO expression, and compare to the observed perturbed cells.
#
#  Key design decisions:
#    • Single inference pass per perturbation — no double loop for T3
#    • Per-cell predictions are computed once, centroid + T3 derived together
#    • No GEARS / torch_geometric / fine-tuning needed
#    • torchtext ABI crash fixed by uninstall + sys.modules stub
#    • Standalone GeneVocab — zero torchtext dependency at runtime
#
#  Install footprint: scGPT (git) + scanpy + anndata + huggingface_hub only
#  Runtime: ~10-25 min on T4/A100 (inference-only)
# ════════════════════════════════════════════════════════════════════════════

# ─── 0. INSTALLS ─────────────────────────────────────────────────────────
import subprocess, sys, os, warnings


def pip(*pkgs, quiet=True):
    args = [sys.executable, "-m", "pip", "install", "--break-system-packages"]
    if quiet:
        args += ["-q"]
    subprocess.check_call(args + list(pkgs))


def pip_uninstall(*pkgs):
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", *pkgs],
        capture_output=True,
    )


import torch
_tv  = torch.__version__.split("+")[0]
_cud = torch.version.cuda.replace(".", "") if torch.version.cuda else "cpu"
print(f"torch {_tv}  cuda {_cud}", flush=True)

print("Installing scGPT (git HEAD) ...", flush=True)
pip("git+https://github.com/bowang-lab/scGPT.git", quiet=False)

print("Installing aux deps ...", flush=True)
pip("huggingface_hub", "scanpy", "anndata")

# Remove torchtext before it can be imported.
# scGPT's setup.cfg pulls it in; libtorchtext.so is ABI-locked to an older
# torch and raises "undefined symbol" on Colab torch >=2.4.
pip_uninstall("torchtext")
print("torchtext uninstalled (ABI fix).\n", flush=True)
warnings.filterwarnings("ignore")

# ─── 1. TORCHTEXT STUB ───────────────────────────────────────────────────
# scGPT internals import `from torchtext.vocab import Vocab` at load time.
# Evict any cached torchtext modules, then inject a lightweight stub tree
# so those imports succeed without touching any compiled extension.

import types as _types

for _k in list(sys.modules.keys()):
    if "torchtext" in _k:
        del sys.modules[_k]


class _VocabStub:
    """Minimal torchtext.vocab.Vocab substitute — satisfies scGPT internals."""
    def __init__(self, stoi=None):
        self._stoi = stoi or {}
        self._itos = {v: k for k, v in self._stoi.items()}
        self._default_index = 0
    def __contains__(self, t):      return t in self._stoi
    def __getitem__(self, t):       return self._stoi.get(t, self._default_index)
    def __len__(self):              return len(self._stoi)
    def set_default_index(self, i): self._default_index = i
    def append_token(self, t):
        if t not in self._stoi:
            i = len(self._stoi)
            self._stoi[t] = i
            self._itos[i] = t
    def get_stoi(self): return self._stoi
    def get_itos(self): return [self._itos.get(i, "<unk>") for i in range(len(self._itos))]
    @staticmethod
    def build_vocab_from_iterator(it, **kw):
        s = {}
        for toks in it:
            for t in toks:
                if t not in s:
                    s[t] = len(s)
        return _VocabStub(s)


def _stub_mod(name):
    return _types.ModuleType(name)

_tt_voc = _stub_mod("torchtext.vocab")
_tt_voc.Vocab = _VocabStub
_tt_voc.build_vocab_from_iterator = _VocabStub.build_vocab_from_iterator
_tt_txt = _stub_mod("torchtext._torchtext")
_tt_txt.Vocab = _VocabStub

for _name, _mod in [
    ("torchtext",            _stub_mod("torchtext")),
    ("torchtext.vocab",      _tt_voc),
    ("torchtext._torchtext", _tt_txt),
    ("torchtext.data",       _stub_mod("torchtext.data")),
    ("torchtext.utils",      _stub_mod("torchtext.utils")),
    ("torchtext.datasets",   _stub_mod("torchtext.datasets")),
    ("torchtext.functional", _stub_mod("torchtext.functional")),
]:
    sys.modules[_name] = _mod

print("torchtext stub injected.", flush=True)

# ─── 2. STANDALONE GeneVocab ─────────────────────────────────────────────
# scGPT's own GeneVocab wraps torchtext — bypassed entirely here.

class GeneVocab:
    """Reads vocab.json directly; zero torchtext dependency."""

    def __init__(self, stoi: dict):
        self._stoi = dict(stoi)
        self._itos = {v: k for k, v in stoi.items()}
        self._default_index = 0

    @classmethod
    def from_file(cls, path: str) -> "GeneVocab":
        import json
        with open(path) as f:
            data = json.load(f)
        stoi = data if isinstance(data, dict) else {t: i for i, t in enumerate(data)}
        return cls(stoi)

    def __contains__(self, t): return t in self._stoi
    def __getitem__(self, t):  return self._stoi.get(t, self._default_index)
    def __len__(self):         return len(self._stoi)
    def set_default_index(self, i): self._default_index = i
    def append_token(self, t):
        if t not in self._stoi:
            i = len(self._stoi)
            self._stoi[t] = i
            self._itos[i] = t
    def get_stoi(self): return self._stoi
    def get_itos(self): return [self._itos.get(i, "<unk>") for i in range(len(self._itos))]
    def lookup_indices(self, toks): return [self[t] for t in toks]
    def lookup_tokens(self, idxs):  return [self._itos.get(i, "<unk>") for i in idxs]

# ─── 3. IMPORTS ──────────────────────────────────────────────────────────
import json
from pathlib import Path
from typing import Dict, Optional

import numpy as np
import scipy.sparse as sp
import scanpy as sc
import torch
import torch.nn as nn
from tqdm import tqdm

# torchtext stub is active — scGPT imports are now safe
from scgpt.model import TransformerGenerator
from scgpt.utils import set_seed

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"numpy  : {np.__version__}")
print(f"torch  : {torch.__version__}\n")

# AMP: use torch.amp on PyTorch >=2.4, fall back to cuda.amp on older
if hasattr(torch.amp, "GradScaler"):
    def _autocast():
        return torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda"))
else:
    def _autocast():
        return torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda"))

# flash-attn detection
try:
    import flash_attn  # noqa: F401
    _USE_FAST = True
    print("flash-attn found  → use_fast_transformer=True", flush=True)
except ImportError:
    _USE_FAST = False
    print("flash-attn absent → use_fast_transformer=False", flush=True)

# ─── 4. CONFIG ───────────────────────────────────────────────────────────
H5AD_PATH      = "/content/K562.h5ad"
CTRL_KEY       = "non-targeting"   # obs[GENE_COL] value for control cells
GENE_COL       = "gene"            # obs column holding perturbation gene name
SUBSAMPLE_FRAC = 0.20              # fraction of cells kept per perturbation
MIN_CELLS      = 5                 # min cells after subsampling to eval a pert
MAX_EVAL_GENES = 1500              # token width per forward pass (T4-safe)
N_CTRL_CELLS   = 200               # ctrl cells used as model input per pert
INFER_BATCH    = 32                # cells per model.forward() call
MAX_T3_CELLS   = 500               # cells per pert for T3 distance matrices
SCGPT_DIR      = "./scGPT_human"
LOAD_PREFIXES  = ["encoder", "value_encoder", "transformer_encoder"]


# ─── 5. DOWNLOAD scGPT_human checkpoint (Google Drive) ───────────────────
# Official distribution is Google Drive — NOT HuggingFace.
# The bowang-lab HF integration was announced but never completed.
# Folder ID from: https://virtualcellmodels.cziscience.com/quickstart/scgpt-quickstart
GDRIVE_FOLDER_ID = "1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y"

if not Path(SCGPT_DIR).exists():
    print("Installing gdown ...", flush=True)
    pip("gdown")
    import gdown
    print("Downloading scGPT_human from Google Drive (~500 MB) ...", flush=True)
    gdown.download_folder(
        id=GDRIVE_FOLDER_ID,
        output=SCGPT_DIR,
        quiet=False,
        use_cookies=False,
    )
    print("Download complete.", flush=True)
else:
    print(f"Using cached checkpoint: {SCGPT_DIR}/", flush=True)

MODEL_FILE = str(Path(SCGPT_DIR) / "best_model.pt")
VOCAB_FILE = str(Path(SCGPT_DIR) / "vocab.json")
# gdown sometimes nests files under a subfolder named after the Drive folder.
# Search recursively for best_model.pt and update SCGPT_DIR if needed.
def _find_file(root, name):
    hits = list(Path(root).rglob(name))
    return str(hits[0].parent) if hits else None

if not Path(MODEL_FILE).exists():
    _found = _find_file(SCGPT_DIR, "best_model.pt")
    if _found:
        SCGPT_DIR = _found
        MODEL_FILE = str(Path(SCGPT_DIR) / "best_model.pt")
        VOCAB_FILE = str(Path(SCGPT_DIR) / "vocab.json")
        print(f"  [auto] relocated checkpoint to: {SCGPT_DIR}", flush=True)
    else:
        raise FileNotFoundError(
            f"best_model.pt not found under {SCGPT_DIR}. "
            "Re-run with the SCGPT_DIR directory deleted to force re-download."
        )
assert Path(MODEL_FILE).exists(), f"Missing {MODEL_FILE}"
assert Path(VOCAB_FILE).exists(), f"Missing {VOCAB_FILE}"


# ─── 6. LOAD & PREPROCESS h5ad ───────────────────────────────────────────
print(f"\n[1/4] Loading {H5AD_PATH} ...", flush=True)
adata = sc.read_h5ad(H5AD_PATH)
print(f"  Loaded    : {adata.shape}", flush=True)

# Stratified 20% subsample per perturbation (same as CPA/GEARS/STATE scripts)
rng  = np.random.default_rng(42)
keep = []
for _gene, _grp in adata.obs.groupby(GENE_COL, observed=True):
    n = max(int(len(_grp) * SUBSAMPLE_FRAC), 1)
    keep.extend(rng.choice(_grp.index, size=min(n, len(_grp)), replace=False).tolist())
adata = adata[keep].copy()
print(f"  Subsampled: {adata.shape}", flush=True)

# Auto-detect raw counts and normalize
_X_check = adata.X[:1000]
if sp.issparse(_X_check):
    _X_check = _X_check.toarray()
_looks_raw = bool(np.nanmax(_X_check) > 50 or
                  np.array_equal(_X_check, _X_check.astype(int)))
if _looks_raw:
    print("  Applying normalize_total(1e4) + log1p ...", flush=True)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
else:
    print("  Data looks log-normalized — skipping normalization.", flush=True)

# Dense float32 matrix for fast numpy slicing
X_dense = (adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)).astype(np.float32)

ctrl_mask_obs = (adata.obs[GENE_COL] == CTRL_KEY).values
n_ctrl_total  = int(ctrl_mask_obs.sum())
print(f"  ctrl cells : {n_ctrl_total}", flush=True)
print(f"  pert cells : {int((~ctrl_mask_obs).sum())}", flush=True)

# ─── 7. VOCAB & EVAL GENE SET ────────────────────────────────────────────
print("\n[2/4] Vocabulary & eval gene set ...", flush=True)

PAD_TOKEN   = "<pad>"
PAD_VALUE   = 0
PERT_PAD_ID = 2

vocab = GeneVocab.from_file(VOCAB_FILE)
for _s in [PAD_TOKEN, "<cls>", "<eoc>"]:
    if _s not in vocab:
        vocab.append_token(_s)
vocab.set_default_index(vocab[PAD_TOKEN])
print(f"  vocab size        : {len(vocab)}", flush=True)

# ── Resolve gene symbols ─────────────────────────────────────────────────
# h5ad var_names may be Ensembl IDs; scGPT vocab uses gene symbols.
# Search var columns for one whose values appear in the vocab.
_SYMBOL_COL_CANDIDATES = [
    "gene_name", "gene_names", "gene_symbols", "symbol", "Gene",
    "feature_name", "gene_short_name", "name", "var_gene_name",
]
_gene_symbol_col = None
for _col in _SYMBOL_COL_CANDIDATES:
    if _col in adata.var.columns:
        _test = adata.var[_col].astype(str).tolist()
        _hits = sum(1 for g in _test[:200] if g in vocab)
        if _hits > 10:          # at least 10/200 match → good column
            _gene_symbol_col = _col
            break

if _gene_symbol_col is not None:
    print(f"  Using adata.var['{_gene_symbol_col}'] as gene symbols for vocab matching.", flush=True)
    all_gene_names = adata.var[_gene_symbol_col].astype(str).tolist()
elif any(g.startswith("ENSG") for g in list(adata.var_names)[:20]):
    # var_names are Ensembl IDs and no symbol column found.
    # Try the 'gene_ids' column which is sometimes the reverse mapping.
    _tried = [c for c in adata.var.columns]
    print(f"  [WARN] var_names look like Ensembl IDs and no symbol col auto-detected.", flush=True)
    print(f"  var columns available: {_tried}", flush=True)
    print(f"  Falling back to var_names — expect zero intersection.", flush=True)
    all_gene_names = list(adata.var_names)
else:
    all_gene_names = list(adata.var_names)

in_vocab       = np.array([g in vocab for g in all_gene_names])
print(f"  h5ad ∩ vocab      : {in_vocab.sum()} / {len(all_gene_names)}", flush=True)

# Select top MAX_EVAL_GENES by ctrl-cell variance from the intersection
# Using ctrl cells only ensures perturbation signal doesn't inflate variance.
ctrl_X         = X_dense[ctrl_mask_obs]           # (n_ctrl, n_h5ad_genes)
inter_idx      = np.where(in_vocab)[0]             # h5ad column positions
ctrl_var       = ctrl_X[:, inter_idx].var(axis=0)
top_idx        = np.argsort(ctrl_var)[::-1][:min(MAX_EVAL_GENES, len(inter_idx))]
eval_h5ad_idx  = np.sort(inter_idx[top_idx])      # sorted for reproducibility
eval_gene_names = [all_gene_names[i] for i in eval_h5ad_idx]
eval_vocab_ids  = np.array([vocab[g] for g in eval_gene_names], dtype=int)
n_eval          = len(eval_gene_names)
# gene_name → position in eval set (for pert_flag lookup)
eval_gene_pos   = {g: i for i, g in enumerate(eval_gene_names)}

# Also build obs-level lookup: what symbol does each pert gene in obs[GENE_COL] correspond to?
# obs[GENE_COL] values may be either symbols or Ensembl IDs depending on the dataset.
# Build a map: obs_pert_name → eval_gene_pos key (symbol)
if _gene_symbol_col is not None:
    # var_names = Ensembl, all_gene_names = symbols
    # obs[GENE_COL] might be Ensembl IDs or symbols — detect which
    _sample_obs = [g for g in adata.obs[GENE_COL].unique()[:20]
                   if g != CTRL_KEY and "+" not in g]
    _obs_in_eval_direct = sum(1 for g in _sample_obs if g in eval_gene_pos)
    _ensg2sym = dict(zip(list(adata.var_names), all_gene_names))
    _obs_in_eval_via_map = sum(1 for g in _sample_obs if _ensg2sym.get(g, g) in eval_gene_pos)
    if _obs_in_eval_direct >= _obs_in_eval_via_map:
        OBS_TO_EVAL = lambda g: g           # obs already uses symbols
        print("  obs[GENE_COL] uses gene symbols — direct lookup.", flush=True)
    else:
        OBS_TO_EVAL = lambda g: _ensg2sym.get(g, g)   # obs uses Ensembl IDs
        print("  obs[GENE_COL] uses Ensembl IDs — mapping via var table.", flush=True)
else:
    OBS_TO_EVAL = lambda g: g

print(f"  eval genes        : {n_eval}", flush=True)

# ── Diagnostic: abort early if intersection is empty ─────────────────────
if n_eval == 0:
    print("\n[FATAL] Zero genes in h5ad ∩ vocab intersection!", flush=True)
    print(f"  Sample h5ad var_names : {all_gene_names[:10]}", flush=True)
    print(f"  Sample vocab keys     : {list(vocab.get_stoi().keys())[:10]}", flush=True)
    raise RuntimeError(
        "h5ad var_names and scGPT vocab share no genes. "
        "Check that h5ad var_names are gene symbols (not Ensembl IDs). "
        "The K562.h5ad must have gene symbol var_names (e.g. 'TP53', not 'ENSG...')."
    )
print(f"  Sample eval genes     : {eval_gene_names[:5]}", flush=True)

# ─── 8. MODEL ────────────────────────────────────────────────────────────
print("\n[3/4] Building TransformerGenerator ...", flush=True)

_cfg_path = Path(SCGPT_DIR) / "args.json"
if _cfg_path.exists():
    with open(_cfg_path) as _f:
        _cfg = json.load(_f)
    EMBSIZE   = _cfg.get("embsize",      512)
    NHEAD     = _cfg.get("nheads",         8)
    D_HID     = _cfg.get("d_hid",        512)
    NLAYERS   = _cfg.get("nlayers",       12)
    N_CLS_LYR = _cfg.get("n_layers_cls",  3)
else:
    EMBSIZE, NHEAD, D_HID, NLAYERS, N_CLS_LYR = 512, 8, 512, 12, 3

model = TransformerGenerator(
    len(vocab), EMBSIZE, NHEAD, D_HID, NLAYERS,
    nlayers_cls=N_CLS_LYR, n_cls=1,
    vocab=vocab,
    dropout=0.0,          # inference only — no dropout
    pad_token=PAD_TOKEN, pad_value=PAD_VALUE, pert_pad_id=PERT_PAD_ID,
    do_mvc=False, cell_emb_style="cls",
    mvc_decoder_style="inner product",
    use_fast_transformer=_USE_FAST,
)

# Partial weight load: encoder + value_encoder + transformer_encoder only
_md = model.state_dict()
_pd = torch.load(MODEL_FILE, map_location="cpu", weights_only=False)
_pd = {k: v for k, v in _pd.items()
       if any(k.startswith(p) for p in LOAD_PREFIXES) and k in _md}
_md.update(_pd)
model.load_state_dict(_md)
model.eval()
model.to(DEVICE)
print(f"  loaded {len(_pd)} / {len(_md)} tensors from scGPT_human", flush=True)

# Pre-built constant tensors (identical for every forward pass)
# Kept on CPU; moved to DEVICE inside the loop (expand is zero-copy)
_GENE_IDS = torch.tensor(eval_vocab_ids, dtype=torch.long).unsqueeze(0)  # (1, n_eval)
_PAD_MASK = torch.zeros(1, n_eval, dtype=torch.bool)                     # (1, n_eval)

# Fixed ctrl cell sample — drawn once, reused for every perturbation
n_ctrl_use      = min(N_CTRL_CELLS, n_ctrl_total)
ctrl_sample_idx = rng.choice(n_ctrl_total, size=n_ctrl_use, replace=False)
ctrl_eval_full  = ctrl_X[:, eval_h5ad_idx]                  # (n_ctrl, n_eval)
ctrl_expr_used  = ctrl_eval_full[ctrl_sample_idx]            # (n_ctrl_use, n_eval)
ctrl_mean_eval  = ctrl_eval_full.mean(axis=0).astype(np.float32)  # (n_eval,)

# ─── 9. SINGLE-PASS INFERENCE + METRICS ──────────────────────────────────
# One loop over perturbations.  For each gene:
#   a) skip if too few cells or gene not in eval set
#   b) run forward pass → per-cell predictions (n_ctrl_use, n_eval)
#   c) derive centroid = mean over ctrl cells
#   d) compute T3 (Energy + MMD) immediately from per-cell arrays
#   e) store centroid for T1/T2 (computed after loop — needs all perts)
#
# This eliminates the second inference loop that the prior version required
# for T3, cutting runtime roughly in half.

print("\n[4/4] Per-perturbation inference + distributional metrics ...", flush=True)

unique_perts_all  = adata.obs[GENE_COL].unique().tolist()
single_gene_perts = [
    g for g in unique_perts_all
    if g != CTRL_KEY and "+" not in g   # skip ctrl + combinatorial KOs
]

pred_centroids = []   # list of (n_eval,) arrays — predicted centroids
true_centroids = []   # list of (n_eval,) arrays — observed centroids
pert_names     = []   # corresponding gene names
energy_list    = []   # per-pert Energy Distance
mmd_list       = []   # per-pert MMD (RBF)

skipped_few  = 0
skipped_oov  = 0

def _t(x: np.ndarray):
    """numpy → float32 CUDA/CPU tensor."""
    return torch.tensor(x, dtype=torch.float32, device=DEVICE)


@torch.no_grad()
def _run_pert(pert_pos: int) -> np.ndarray:
    """
    Run batched inference for one perturbation gene.
    Returns per-cell predictions: float32 numpy (n_ctrl_use, n_eval).
    pert_pos : column index of the KO gene in the eval gene set.
    """
    chunks = []
    for start in range(0, n_ctrl_use, INFER_BATCH):
        end  = min(start + INFER_BATCH, n_ctrl_use)
        B    = end - start
        expr = torch.tensor(ctrl_expr_used[start:end], dtype=torch.float32)

        # pert_flags: 0 everywhere, 1 at the target gene column
        pflg = torch.zeros(B, n_eval, dtype=torch.long)
        pflg[:, pert_pos] = 1

        with _autocast():
            out = model(
                _GENE_IDS.expand(B, -1).to(DEVICE),
                expr.to(DEVICE),
                pflg.to(DEVICE),
                src_key_padding_mask=_PAD_MASK.expand(B, -1).to(DEVICE),
                CLS=False, CCE=False, MVC=False, ECS=False,
            )
        chunks.append(out["mlm_output"].float().cpu().numpy())

    return np.concatenate(chunks, axis=0)   # (n_ctrl_use, n_eval)


# ── Pre-flight: check perturbation gene names match eval set ─────────────
_sample_perts = [g for g in single_gene_perts if g != CTRL_KEY][:20]
_in_eval = sum(1 for g in _sample_perts if OBS_TO_EVAL(g) in eval_gene_pos)
print(f"  Pre-flight: {_in_eval}/{len(_sample_perts)} sample perts found in eval gene set", flush=True)
if _in_eval == 0 and len(_sample_perts) > 0:
    print(f"  Sample pert names : {_sample_perts[:5]}", flush=True)
    print(f"  Sample eval genes : {eval_gene_names[:5]}", flush=True)
    raise RuntimeError(
        "None of the perturbation gene names appear in the eval gene set. "
        "This usually means h5ad obs[GENE_COL] uses a different naming "
        "convention than var_names (e.g. lowercase vs uppercase). "
        f"Check GENE_COL='{GENE_COL}' and sample values above."
    )

for gene in tqdm(single_gene_perts, ncols=72, desc="inference+T3"):

    # ── gate: enough true cells? ──────────────────────────────────────────
    pert_mask_i  = (adata.obs[GENE_COL] == gene).values
    n_pert_cells = int(pert_mask_i.sum())
    if n_pert_cells < MIN_CELLS:
        skipped_few += 1
        continue

    # ── gate: gene in eval set? ───────────────────────────────────────────
    pert_pos = eval_gene_pos.get(OBS_TO_EVAL(gene), -1)
    if pert_pos < 0:
        skipped_oov += 1
        continue

    # ── forward pass (single run, used for both centroid and T3) ─────────
    pred_cells = _run_pert(pert_pos)                       # (n_ctrl_use, n_eval)
    true_cells = X_dense[pert_mask_i][:, eval_h5ad_idx]   # (n_pert, n_eval)

    # ── centroids (stored for T1/T2) ─────────────────────────────────────
    pred_centroids.append(pred_cells.mean(axis=0))
    true_centroids.append(true_cells.mean(axis=0))
    pert_names.append(gene)

    # ── T3: Energy Distance + MMD (RBF) ──────────────────────────────────
    p_arr = pred_cells.copy()
    t_arr = true_cells

    if p_arr.shape[0] < 2 or t_arr.shape[0] < 2:
        energy_list.append(0.0); mmd_list.append(0.0)
        continue

    # Subsample if needed
    if p_arr.shape[0] > MAX_T3_CELLS:
        p_arr = p_arr[rng.choice(p_arr.shape[0], MAX_T3_CELLS, replace=False)]
    if t_arr.shape[0] > MAX_T3_CELLS:
        t_arr = t_arr[rng.choice(t_arr.shape[0], MAX_T3_CELLS, replace=False)]

    pt = _t(p_arr); tt = _t(t_arr)
    np_ = pt.shape[0]; nt_ = tt.shape[0]

    dpp = torch.cdist(pt, pt)
    dtt = torch.cdist(tt, tt)
    dpt = torch.cdist(pt, tt)

    mask_pp = ~torch.eye(np_, dtype=torch.bool, device=DEVICE)
    mask_tt = ~torch.eye(nt_, dtype=torch.bool, device=DEVICE)
    off_pp  = dpp[mask_pp].mean()
    off_tt  = dtt[mask_tt].mean()

    # Energy Distance: 2·E[d(P,T)] − E[d(P,P)] − E[d(T,T)]
    energy_list.append(max((2.0 * dpt.mean() - off_pp - off_tt).item(), 0.0))

    # MMD (RBF kernel, median heuristic for bandwidth)
    # σ² = median( ||t_i − t_j||² ) / 2   for i ≠ j
    dtt_sq = dtt.pow(2)
    sigma2 = dtt_sq[mask_tt].median() / 2.0
    if sigma2 < 1e-10:
        sigma2 = torch.tensor(1.0, device=DEVICE)
    Kpp = torch.exp(-dpp.pow(2) / (2.0 * sigma2)).mean()
    Ktt = torch.exp(-dtt_sq    / (2.0 * sigma2)).mean()
    Kpt = torch.exp(-dpt.pow(2) / (2.0 * sigma2)).mean()
    mmd_list.append(max((Kpp - 2.0 * Kpt + Ktt).item(), 0.0))

    # Free GPU tensors immediately — keeps VRAM flat across 2000 perts
    del pt, tt, dpp, dtt, dpt
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# ── Aggregate ─────────────────────────────────────────────────────────────
n_perts = len(pert_names)
print(f"\n  Evaluated   : {n_perts} perturbations", flush=True)
print(f"  Skipped (< {MIN_CELLS} cells)     : {skipped_few}", flush=True)
print(f"  Skipped (not in eval set) : {skipped_oov}", flush=True)

if n_perts == 0:
    print("\n[FATAL] No perturbations were evaluated.", flush=True)
    if skipped_oov == len(single_gene_perts):
        print("  ALL perts were skipped because their gene name was not in the eval set.", flush=True)
        print(f"  Sample pert names  : {single_gene_perts[:5]}", flush=True)
        print(f"  Sample eval genes  : {eval_gene_names[:5]}", flush=True)
        print("  Hint: GENE_COL obs values must match h5ad var_names exactly.", flush=True)
    elif skipped_few == len(single_gene_perts):
        print(f"  ALL perts had < {MIN_CELLS} cells after subsampling.", flush=True)
        print(f"  Lower MIN_CELLS (currently {MIN_CELLS}) or SUBSAMPLE_FRAC (currently {SUBSAMPLE_FRAC}).", flush=True)
    raise RuntimeError("Zero perturbations evaluated — see diagnostics above.")

pred_mat     = np.stack(pred_centroids).astype(np.float32)  # (P, n_eval)
true_mat     = np.stack(true_centroids).astype(np.float32)  # (P, n_eval)
energy_dist  = float(np.mean(energy_list))
mmd_val      = float(np.mean(mmd_list))

# ─── 10. TIER 1 + TIER 2 METRICS ─────────────────────────────────────────
print("\nComputing T1 / T2 metrics ...", flush=True)

D = DEVICE

pred_c  = _t(pred_mat)
true_c  = _t(true_mat)
ctrl_t  = _t(ctrl_mean_eval)

pred_delta = pred_c - ctrl_t.unsqueeze(0)   # (P, n_eval)
true_delta = true_c - ctrl_t.unsqueeze(0)

def _batch_pearson(a, b):
    a = a - a.mean(1, keepdim=True)
    b = b - b.mean(1, keepdim=True)
    return (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-8)

# ── Tier 1 ────────────────────────────────────────────────────────────────
dist_pt  = torch.cdist(pred_c, true_c)                           # (P, P)
nearest  = dist_pt.argmin(dim=1)
ca       = (nearest == torch.arange(n_perts, device=D)).float().mean().item()

# Systema PDS: d_self / (d_self + d_cross + ε)
d_self   = dist_pt.diagonal()                                    # (P,)
_off     = ~torch.eye(n_perts, dtype=torch.bool, device=D)
d_cross  = (dist_pt * _off).sum(1) / _off.float().sum(1)        # (P,)
pds      = (d_self / (d_self + d_cross + 1e-8)).mean().item()

systema_r = _batch_pearson(pred_delta, true_delta).mean().item()

# ── Tier 2 ────────────────────────────────────────────────────────────────
sig_mask  = true_delta.abs() > 0.1
dir_match = ((pred_delta > 0) == (true_delta > 0)) & sig_mask
dir_acc   = (dir_match.float().sum() /
             sig_mask.float().sum().clamp(min=1)).item()

TOP_DE = 50
de_r_list, de_j_list = [], []
for i in range(n_perts):
    td, pd_i = true_delta[i], pred_delta[i]
    k        = min(TOP_DE, n_eval)
    top_true = td.abs().topk(k).indices
    top_pred = pd_i.abs().topk(k).indices
    de_r_list.append(
        _batch_pearson(pd_i[top_true].unsqueeze(0),
                       td[top_true].unsqueeze(0)).item()
    )
    ts = set(top_true.cpu().tolist())
    ps = set(top_pred.cpu().tolist())
    de_j_list.append(len(ts & ps) / len(ts | ps) if (ts | ps) else 0.0)

pearson_de = float(np.nanmean(de_r_list))
jaccard_de = float(np.mean(de_j_list))

# ─── 11. RESULTS ─────────────────────────────────────────────────────────
SEP = "=" * 66
print(f"\n{SEP}")
print(f"  scGPT · K562 · Zero-Shot (scGPT_human, no fine-tuning)")
print(f"  {n_perts} perts | {n_eval} eval genes | log1p-norm | single-pass inference")
print(f"  {'Tier  Metric':<44} {'Value':>8}  Dir")
print(f"  {'-'*62}")


def _row(tag, name, val, hi):
    print(f"  [{tag}] {name:<42} {val:>8.4f}  {'up' if hi else 'down'}")


_row("T1", "Centroid Accuracy (CA)",        ca,          hi=True)
_row("T1", "Profile Distance Score (PDS)",  pds,         hi=False)
_row("T1", "Systema Pearson Delta",         systema_r,   hi=True)
_row("T2", "Directional Accuracy",          dir_acc,     hi=True)
_row("T2", "Pearson Delta DE Top-50",       pearson_de,  hi=True)
_row("T2", "Jaccard DE Top-50",             jaccard_de,  hi=True)
_row("T3", "Energy Distance",               energy_dist, hi=False)
_row("T3", "MMD (RBF kernel)",              mmd_val,     hi=False)
print(SEP)
print("Done.")

torch 2.10.0  cuda 128
Installing scGPT (git HEAD) ...
Installing aux deps ...
torchtext uninstalled (ABI fix).

torchtext stub injected.
Device : cuda
numpy  : 2.0.2
torch  : 2.10.0+cu128

flash-attn absent → use_fast_transformer=False
Installing gdown ...


Retrieving folder contents


Processing file 1hh2zGKyWAx3DyovD30GStZ3QlzmSqdk1 args.json
Processing file 14AebJfGOUF047Eg40hk57HCtrb0fyDTm best_model.pt
Processing file 1H3E_MJ-Dl36AQV6jLbna2EdvgPaqvqcC vocab.json


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1hh2zGKyWAx3DyovD30GStZ3QlzmSqdk1
To: /content/scGPT_human/args.json
100%|██████████| 1.30k/1.30k [00:00<00:00, 3.11MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=14AebJfGOUF047Eg40hk57HCtrb0fyDTm
From (redirected): https://drive.google.com/uc?id=14AebJfGOUF047Eg40hk57HCtrb0fyDTm&confirm=t&uuid=57ccb81d-166c-4028-8e40-bab2076d11b0
To: /content/scGPT_human/best_model.pt
100%|██████████| 205M/205M [00:01<00:00, 106MB/s]
Downloading...
From: https://drive.google.com/uc?id=1H3E_MJ-Dl36AQV6jLbna2EdvgPaqvqcC
To: /content/scGPT_human/vocab.json
100%|██████████| 1.32M/1.32M [00:00<00:00, 180MB/s]

Download complete.

[1/4] Loading /content/K562.h5ad ...



Download completed


  Loaded    : (310385, 8563)
  Subsampled: (61244, 8563)
  Applying normalize_total(1e4) + log1p ...
  ctrl cells : 2138
  pert cells : 59106

[2/4] Vocabulary & eval gene set ...
  vocab size        : 60697
  Using adata.var['gene_name'] as gene symbols for vocab matching.
  h5ad ∩ vocab      : 8324 / 8563
  obs[GENE_COL] uses gene symbols — direct lookup.
  eval genes        : 1500
  Sample eval genes     : ['NOC2L', 'MRPL20', 'ATAD3A', 'SSU72', 'GNB1']

[3/4] Building TransformerGenerator ...
  loaded 129 / 176 tensors from scGPT_human

[4/4] Per-perturbation inference + distributional metrics ...
  Pre-flight: 6/20 sample perts found in eval gene set


inference+T3: 100%|█████████████████| 2057/2057 [04:16<00:00,  8.02it/s]


  Evaluated   : 560 perturbations
  Skipped (< 5 cells)     : 54
  Skipped (not in eval set) : 1443

Computing T1 / T2 metrics ...



  scGPT · K562 · Zero-Shot (scGPT_human, no fine-tuning)
  560 perts | 1500 eval genes | log1p-norm | single-pass inference
  Tier  Metric                                    Value  Dir
  --------------------------------------------------------------
  [T1] Centroid Accuracy (CA)                       0.0018  up
  [T1] Profile Distance Score (PDS)                 0.4998  down
  [T1] Systema Pearson Delta                        0.0518  up
  [T2] Directional Accuracy                         0.5799  up
  [T2] Pearson Delta DE Top-50                      0.1302  up
  [T2] Jaccard DE Top-50                            0.0644  up
  [T3] Energy Distance                             40.2591  down
  [T3] MMD (RBF kernel)                             1.0340  down
Done.


restart here

In [ ]:
# =============================================================================
# Cell2Sentence (C2S-Scale-Gemma-2-2B) — K562 Comprehensive Eval (v4.0)
# =============================================================================
# • CHANGED: 27B → 2B model (5× faster loading, 14× faster inference)
# • INCREASED: MAX_PERTURBATIONS 25 → 200
# • INCREASED: MAX_NEW_TOKENS 300 → 600
# • INCREASED: SUBSAMPLE 5% → 15%
# • FIXED: CUDA assert issues (proper bfloat16 + updated BNB)
# • TARGET: ≤15 minutes total runtime with comprehensive evaluation
# Prerequisites: Upload /content/K562.h5ad
# =============================================================================

import subprocess, sys, os, shutil, gc, time, warnings
import numpy as np
import torch
import pandas as pd
import scanpy as sc
from scipy.spatial.distance import cdist
from sklearn.metrics import r2_score
from collections import defaultdict
from tqdm import tqdm
from sklearn.linear_model import LinearRegression

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION - OPTIMIZED FOR 2B MODEL
# =============================================================================
C2S_DIR           = "/tmp/vandijklab_c2s"
DATA_PATH         = "/content/K562.h5ad"
MODEL_NAME        = "vandijklab/C2S-Scale-Gemma-2-2B"  # CHANGED: 27B → 2B
CTRL_LABEL        = "non-targeting"
SEED              = 42
TOP_K_DE          = 50
TOP_K_GENES       = 200
SUBSAMPLE         = 0.15              # INCREASED: 5% → 15% (more cells for better stats)
MAX_NEW_TOKENS    = 600               # INCREASED: 300 → 600 (better generation)
MAX_PERTURBATIONS = 200               # INCREASED: 25 → 200 (comprehensive eval)
BATCH_SIZE        = 1
EVAL_SAMPLE_CELLS = 50                # Cells per perturbation for evaluation

def _pip(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + list(packages))

# =============================================================================
# STEP 1-6: SETUP & DEPENDENCIES
# =============================================================================
print("STEP 1-6: Environment Setup...")
start_time = time.time()

if os.path.exists(C2S_DIR):
    shutil.rmtree(C2S_DIR)

subprocess.check_call([
    "git", "clone", "-q",
    "https://github.com/vandijklab/cell2sentence.git",
    C2S_DIR
])

if C2S_DIR not in sys.path:
    sys.path.insert(0, C2S_DIR)

# Install dependencies (with version checks)
try:
    import transformers
    import bitsandbytes
    print(f"  ✅ transformers: {transformers.__version__}")
    print(f"  ✅ bitsandbytes: {bitsandbytes.__version__}")
except ImportError:
    _pip("transformers>=4.45.0", "accelerate>=0.34.0", "bitsandbytes>=0.43.0")
    _pip("cell2sentence==1.1.0", "anndata>=0.10.0", "scanpy>=1.10.0")
    _pip("scikit-learn", "pandas", "scipy", "tqdm")

# Verify CUDA
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available. Please enable GPU runtime.")

print(f"  ✅ CUDA: {torch.cuda.get_device_name(0)}")
print(f"  ✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# =============================================================================
# STEP 7: LOAD 2B MODEL (4-BIT)
# =============================================================================
print("\nSTEP 7: Loading C2S-Scale-Gemma-2-2B (4-bit)...")
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Proper config for Gemma-2
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # Critical for Gemma-2
    bnb_4bit_use_double_quant=True,
    llm_int8_threshold=6.0,
)

gc.collect()
torch.cuda.empty_cache()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    padding_side="left",
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model with proper settings
print("  Loading model weights (~2-3 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

model.eval()
print(f"  ✅ Model loaded on {model.device}")
print(f"  ✅ Model dtype: {model.dtype}")

# =============================================================================
# STEP 8-12: DATA PREPARATION
# =============================================================================
print("\nSTEP 8-12: Loading + preparing data...")
prep_start = time.time()

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

adata = sc.read_h5ad(DATA_PATH)
print(f"  Raw data shape: {adata.shape}")

# Ensure perturbation column
if "perturbation" not in adata.obs.columns:
    if "gene" in adata.obs.columns:
        adata.obs["perturbation"] = adata.obs["gene"].astype(str)
    else:
        raise ValueError("No perturbation column found")

# Filter: >=5 cells per perturbation
vc = adata.obs["perturbation"].value_counts()
adata = adata[adata.obs["perturbation"].isin(vc[vc >= 5].index)].copy()
print(f"  After min-cell filter: {adata.shape}")

# Select TOP perturbations by cell count
pert_counts = adata.obs["perturbation"].value_counts()
pert_counts = pert_counts[pert_counts.index != CTRL_LABEL]
top_perts = pert_counts.head(MAX_PERTURBATIONS).index.tolist()
print(f"  Selected {len(top_perts)} perturbations for evaluation")

# Filter to selected perturbations + control
keep_perts = top_perts + [CTRL_LABEL]
adata = adata[adata.obs["perturbation"].isin(keep_perts)].copy()

# Subsample (15% for better statistics)
rng = np.random.default_rng(SEED)
keep = []
for _, grp in adata.obs.groupby("perturbation"):
    n = min(EVAL_SAMPLE_CELLS, max(2, int(len(grp) * SUBSAMPLE)))
    keep.extend(rng.choice(grp.index, n, replace=False))
adata = adata[keep].copy()
print(f"  After subsampling ({SUBSAMPLE*100}%): {adata.shape}")

# Dense conversion
if hasattr(adata.X, "toarray"):
    adata.X = adata.X.toarray()

# DEGs with proper normalization
print("  Computing DEGs...")
adata_log = adata.copy()
sc.pp.log1p(adata_log)
sc.tl.rank_genes_groups(
    adata_log,
    groupby="perturbation",
    reference=CTRL_LABEL,
    method="t-test",
    n_genes=adata.n_vars
)

rgg = adata_log.uns["rank_genes_groups"]
adata.uns["rank_genes_groups_cov"] = {
    f"K562_{g}": list(rgg["names"][g])
    for g in rgg["names"].dtype.names
    if g != CTRL_LABEL
}

prep_end = time.time()
print(f"  ✅ Data ready in {prep_end - prep_start:.2f}s")

# =============================================================================
# STEP 13: PREDICTION + RECONSTRUCTION
# =============================================================================
print("\nSTEP 13: Zero-shot prediction...")
adata.layers["X_true"] = adata.X.copy()

# Control template
ctrl_mask = adata.obs["perturbation"] == CTRL_LABEL
ctrl_expr = adata[ctrl_mask].X.mean(axis=0).flatten()
top_idx = np.argsort(-ctrl_expr)[:TOP_K_GENES]
template = " ".join(adata.var_names[i] for i in top_idx)

# Power-law fit
ranks = np.argsort(-ctrl_expr)[:TOP_K_GENES]
log_expr = np.log10(ctrl_expr[ranks] + 1e-8)
log_rank = np.log10(1 + np.arange(TOP_K_GENES))
reg = LinearRegression().fit(log_rank.reshape(-1, 1), log_expr)
slope, intercept = reg.coef_[0], reg.intercept_
print(f"  Power-law: slope={slope:.4f}, intercept={intercept:.4f}")

# Generate predictions
pred_sentences = {}
unique_perts = [p for p in top_perts if p != CTRL_LABEL]
print(f"  Generating for {len(unique_perts)} perturbations...")

gc.collect()
torch.cuda.empty_cache()

inference_start = time.time()
for i, condition in enumerate(tqdm(unique_perts, desc="Generating")):
    prompt = (
        f"Given the following cell sentence of {TOP_K_GENES} expressed genes "
        f"representing a cell's basal state, predict the cell sentence after "
        f"applying the perturbation: {condition}.\n"
        f"Control cell sentence: {template}\n"
        f"Perturbed cell sentence:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            temperature=None,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sentence = generated.replace(prompt, "").strip()
    pred_sentences[condition] = sentence

    del inputs, outputs
    if i % 20 == 0:
        torch.cuda.empty_cache()

inference_end = time.time()
inference_time = inference_end - inference_start
print(f"  ✅ Inference completed in {inference_time:.2f}s")
print(f"  Average: {inference_time/len(unique_perts):.2f}s per perturbation")

# Reconstruct
print("  Reconstructing sentence → matrix...")
recon_start = time.time()

pred_X_list = []
vocab = list(adata.var_names)
vocab_set = set(vocab)

for i, row in tqdm(enumerate(adata.obs.itertuples()), desc="Reconstructing", total=adata.n_obs):
    cond = row.perturbation

    if cond == CTRL_LABEL:
        pred_X_list.append(adata.X[i].copy())
        continue

    sentence = pred_sentences.get(cond, "")
    genes = [g.strip() for g in sentence.split() if g.strip() in vocab_set][:TOP_K_GENES]
    expr = np.zeros(adata.n_vars)

    for rank, gene in enumerate(genes):
        idx = vocab.index(gene)
        expr[idx] = 10 ** (slope * np.log10(1 + rank) + intercept)

    expr += np.random.normal(0, 1e-6, size=expr.shape)
    pred_X_list.append(expr)

adata.layers["C2S_pred"] = np.array(pred_X_list)
recon_end = time.time()
print(f"  ✅ Reconstruction in {recon_end - recon_start:.2f}s")

# =============================================================================
# STEP 14-15: METRICS EVALUATION (3-Tier)
# =============================================================================
print("\nSTEP 14-15: Computing metrics...")
eval_start = time.time()

def _pearson(x, y):
    if np.var(x) < 1e-8 or np.var(y) < 1e-8:
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])

def _directional_accuracy(true_diff, pred_diff):
    mask = np.abs(true_diff) > 0.1
    if not mask.any():
        return 0.0
    return float(np.mean(np.sign(true_diff[mask]) == np.sign(pred_diff[mask])))

def _jaccard_topk(true_diff, pred_diff, k=TOP_K_DE):
    true_top = set(np.argsort(np.abs(true_diff))[-k:])
    pred_top = set(np.argsort(np.abs(pred_diff))[-k:])
    union = true_top | pred_top
    return len(true_top & pred_top) / len(union) if union else 0.0

def _energy_dist(x, y, n=300):
    rng2 = np.random.default_rng(SEED)
    p = x[rng2.choice(len(x), min(n, len(x)), replace=False)]
    q = y[rng2.choice(len(y), min(n, len(y)), replace=False)]
    dxy = cdist(p, q).mean()
    dxx = cdist(p, p).mean()
    dyy = cdist(q, q).mean()
    return max(2 * dxy - dxx - dyy, 0.0)

ctrl_mu = adata[ctrl_mask].X.mean(axis=0).flatten()
metrics = defaultdict(list)
perts = [p for p in top_perts if p != CTRL_LABEL]

for pert in tqdm(perts, desc="Evaluating"):
    idx = adata.obs["perturbation"] == pert
    X_true = adata.X[idx]
    X_pred = adata.layers["C2S_pred"][idx]

    true_mu = X_true.mean(axis=0)
    pred_mu = X_pred.mean(axis=0)
    true_diff = true_mu - ctrl_mu
    pred_diff = pred_mu - ctrl_mu

    deg_names = adata.uns["rank_genes_groups_cov"][f"K562_{pert}"][:TOP_K_DE]
    deg_idx = [vocab.index(g) for g in deg_names if g in vocab_set]

    if not deg_idx:
        continue

    # Tier 1: Global similarity
    metrics["T1_CA"].append(
        float(np.dot(true_mu, pred_mu) / (np.linalg.norm(true_mu) * np.linalg.norm(pred_mu) + 1e-8))
    )
    metrics["T1_PDS"].append(_pearson(true_diff[deg_idx], pred_diff[deg_idx]))
    metrics["T1_R2"].append(r2_score(true_mu, pred_mu))

    # Tier 2: DEG-level accuracy
    metrics["T2_DA"].append(_directional_accuracy(true_diff[deg_idx], pred_diff[deg_idx]))
    metrics["T2_Jaccard"].append(_jaccard_topk(np.abs(true_diff), np.abs(pred_diff)))

    # Tier 3: Distribution matching
    metrics["T3_Energy"].append(_energy_dist(X_true[:, deg_idx], X_pred[:, deg_idx]))

eval_end = time.time()
print(f"  ✅ Evaluation in {eval_end - eval_start:.2f}s")

# =============================================================================
# FINAL REPORT
# =============================================================================
total_time = time.time() - start_time
SEP = "=" * 70
print(f"\n{SEP}")
print("  🧬 C2S-Scale-Gemma-2-2B · K562 · Comprehensive Evaluation")
print(f"  {len(perts)} perturbations | {SUBSAMPLE*100}% subsample | {MAX_NEW_TOKENS} tokens")
print(SEP)
print(f"  [T1] Cosine Accuracy:        {np.mean(metrics['T1_CA']):.4f} ± {np.std(metrics['T1_CA']):.4f}")
print(f"  [T1] Pearson Delta:          {np.mean(metrics['T1_PDS']):.4f} ± {np.std(metrics['T1_PDS']):.4f}")
print(f"  [T1] R-squared:              {np.mean(metrics['T1_R2']):.4f} ± {np.std(metrics['T1_R2']):.4f}")
print(f"  [T2] Directional Accuracy:   {np.mean(metrics['T2_DA']):.4f} ± {np.std(metrics['T2_DA']):.4f}")
print(f"  [T2] Jaccard Index:          {np.mean(metrics['T2_Jaccard']):.4f} ± {np.std(metrics['T2_Jaccard']):.4f}")
print(f"  [T3] Energy Distance:        {np.mean(metrics['T3_Energy']):.4f} ± {np.std(metrics['T3_Energy']):.4f}")
print(SEP)
print(f"  ⏱️  Model Load:      {(prep_start - start_time)/60:.2f} min")
print(f"  ⏱️  Data Prep:       {(prep_end - prep_start):.2f} s")
print(f"  ⏱️  Inference:       {inference_time/60:.2f} min")
print(f"  ⏱️  Reconstruction:  {(recon_end - recon_start):.2f} s")
print(f"  ⏱️  Evaluation:      {(eval_end - eval_start):.2f} s")
print(f"  ⏱️  TOTAL:           {total_time/60:.2f} min")
if total_time <= 900:
    print(f"  ✅ TARGET MET: Completed within 15 minutes!")
else:
    print(f"  ⚠️  Exceeded 15 min target")
print(SEP)

# Save results
results_df = pd.DataFrame(metrics)
results_df.to_csv("/content/c2s_comprehensive_eval_results.csv", index=False)
print(f"  📁 Results: /content/c2s_comprehensive_eval_results.csv")

# Save timing
timing = {
    'model_load': prep_start - start_time,
    'data_prep': prep_end - prep_start,
    'inference': inference_time,
    'reconstruction': recon_end - recon_start,
    'evaluation': eval_end - eval_start,
    'total': total_time
}
pd.DataFrame([timing]).to_csv("/content/c2s_timing.csv", index=False)
print(f"  📁 Timing: /content/c2s_timing.csv")

# Save predictions
pred_df = pd.DataFrame(list(pred_sentences.items()), columns=['perturbation', 'predicted_sentence'])
pred_df.to_csv("/content/c2s_predictions.csv", index=False)
print(f"  📁 Predictions: /content/c2s_predictions.csv")

print(f"\n  ✅ Evaluation complete!")

STEP 1-6: Environment Setup...
  ✅ transformers: 5.3.0
  ✅ bitsandbytes: 0.49.2
  ✅ CUDA: NVIDIA A100-SXM4-80GB
  ✅ VRAM: 85.09 GB

STEP 7: Loading C2S-Scale-Gemma-2-2B (4-bit)...
  Loading model weights (~2-3 minutes)...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

  ✅ Model loaded on cuda:0
  ✅ Model dtype: torch.bfloat16

STEP 8-12: Loading + preparing data...
  Raw data shape: (310385, 8563)
  After min-cell filter: (310385, 8563)
  Selected 200 perturbations for evaluation
  After subsampling (15.0%): (9565, 8563)
  Computing DEGs...
  ✅ Data ready in 19.86s

STEP 13: Zero-shot prediction...
  Power-law: slope=-0.7821, intercept=2.8653
  Generating for 200 perturbations...


Generating: 100%|██████████| 200/200 [1:48:55<00:00, 32.68s/it]


  ✅ Inference completed in 6535.09s
  Average: 32.68s per perturbation
  Reconstructing sentence → matrix...


Reconstructing: 100%|██████████| 9565/9565 [00:03<00:00, 3112.81it/s]


  ✅ Reconstruction in 3.28s

STEP 14-15: Computing metrics...


Evaluating: 100%|██████████| 200/200 [00:01<00:00, 125.24it/s]

  ✅ Evaluation in 1.61s

  🧬 C2S-Scale-Gemma-2-2B · K562 · Comprehensive Evaluation
  200 perturbations | 15.0% subsample | 600 tokens
  [T1] Cosine Accuracy:        0.0089 ± 0.0648
  [T1] Pearson Delta:          -0.9035 ± 0.1048
  [T1] R-squared:              -0.0585 ± 0.1120
  [T2] Directional Accuracy:   0.0131 ± 0.0156
  [T2] Jaccard Index:          0.4863 ± 0.1475
  [T3] Energy Distance:        93.7840 ± 132.7127
  ⏱️  Model Load:      0.12 min
  ⏱️  Data Prep:       19.86 s
  ⏱️  Inference:       108.92 min
  ⏱️  Reconstruction:  3.28 s
  ⏱️  Evaluation:      1.61 s
  ⏱️  TOTAL:           109.46 min
  ⚠️  Exceeded 15 min target
  📁 Results: /content/c2s_comprehensive_eval_results.csv
  📁 Timing: /content/c2s_timing.csv
  📁 Predictions: /content/c2s_predictions.csv

  ✅ Evaluation complete!


restart here

In [5]:
# =============================================================================
# CPA K562 — FINAL PRODUCTION PIPELINE (v8.0 — SENIOR ENGINEER AUDITED)
#
# Based on: https://github.com/theislab/cpa
# All Steps 1-16 included for complete reproducibility
# =============================================================================
import subprocess, sys, os, re, shutil, importlib.util, warnings, json
from collections import defaultdict
import numpy as np
import torch
import pandas as pd

# =============================================================================
# CONFIGURATION
# =============================================================================
CPA_DIR       = "/tmp/theislab_cpa"
PRETRAINED_PT = os.path.join("./pretrained_cpa_k562", "k562_model.pt")
MODEL_DIR     = "./pretrained_cpa_k562"
DATA_PATH     = "/content/K562.h5ad"
CTRL_LABEL    = "non-targeting"
SEED          = 42
TOP_K_DE      = 50
SUBSAMPLE     = 0.20
KANG_MODEL_ID = "1IVDsxkCZZlU5MCyiu0MKyAwzeEd_yV4B"

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + list(pkgs))

def _pip_no_upgrade(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

# =============================================================================
# STEP 1 — Clone CPA
# =============================================================================
print("=" * 70)
print("STEP 1/16: Cloning CPA repository...")
if os.path.exists(CPA_DIR):
    shutil.rmtree(CPA_DIR)
subprocess.check_call(["git", "clone", "-q", "https://github.com/theislab/cpa.git", CPA_DIR])
print(f"  Cloned → {CPA_DIR}")

# =============================================================================
# STEP 2 — Install dependencies
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2/16: Installing dependencies...")
_NP_VER = np.__version__
print(f"  NumPy: {_NP_VER} | PyTorch: {torch.__version__}")
_pip("anndata>=0.10.0,<0.13.0")
_pip(f"numpy=={_NP_VER}")
_pip("numba>=0.60.0")
_pip("scanpy>=1.10.0,<1.11.0")
_pip("scvi-tools>=1.0.0,<1.5.0")
_pip("lightning>=2.2.0,<2.4.0")
_pip("pytorch-lightning>=2.2.0,<2.4.0")
_pip("pandas", "tqdm", "gdown")
_pip_no_upgrade("scikit-learn")
_pip("rdkit", "adjustText", "seaborn")
_pip("pybiomart")
print("  All dependencies installed successfully")

# =============================================================================
# STEP 3 — Clear module cache
# =============================================================================
print("\n" + "=" * 70)
print("STEP 3/16: Clearing module cache...")
_CLEAR = ["anndata","scvi","scanpy","cpa","lightning","jax","flax",
          "pytorch_lightning","torchmetrics","numba"]
_n = sum(1 for k in list(sys.modules) if any(k == m or k.startswith(m+".") for m in _CLEAR)
         for _ in [sys.modules.pop(k, None)])
print(f"  Cleared {_n} modules")

# =============================================================================
# STEP 4 — Pre-patch scvi-tools
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4/16: Pre-patching scvi-tools...")

def _get_scvi_path():
    spec = importlib.util.find_spec("scvi")
    return spec.submodule_search_locations[0] if spec and spec.submodule_search_locations else None

def _restore_backups(path):
    for root, _, files in os.walk(path or ""):
        for f in files:
            if f.endswith(".bak_cpa"):
                shutil.copy(os.path.join(root, f), os.path.join(root, f[:-len(".bak_cpa")]))

def _patch_types(p):
    path = next((os.path.join(p, f) for f in ["_types.py", "scvi/_types.py"]
                 if os.path.exists(os.path.join(p, f))), None)
    if not path: return
    bak = path + ".bak_cpa"
    if not os.path.exists(bak): shutil.copy(path, bak)
    with open(path, "w") as f:
        f.write('''from typing import Union
import torch
Tensor = torch.Tensor
Number = Union[int, float]
class MinifiedDataType:
    LATENT_POSTERIOR = "latent_posterior_parameters"
    LATENT_POSTERIOR_WITH_COUNTS = "latent_posterior_parameters_with_counts"
try:
    import anndata as _ad
    AnnOrMuData = Union[_ad.AnnData]
except: AnnOrMuData = object
''')
    print("  Patched: _types.py")

def _patch_negative_binomial(p):
    path = next((os.path.join(p, f) for f in ["distributions/_negative_binomial.py",
                "scvi/distributions/_negative_binomial.py"]
                 if os.path.exists(os.path.join(p, f))), None)
    if not path: return
    with open(path, "r", encoding="utf-8") as f: lines = f.readlines()
    new_lines, patched = [], False
    for i, line in enumerate(lines):
        new_lines.append(line)
        if line.strip().startswith("except") and ":" in line:
            j = i + 1
            while j < len(lines) and not lines[j].strip(): j += 1
            if j >= len(lines) or not lines[j].strip():
                indent = len(line) - len(line.lstrip())
                new_lines.append(" " * (indent + 4) + "pass  # [CPA]\n")
                patched = True
    if patched:
        bak = path + ".bak_cpa"
        if not os.path.exists(bak): shutil.copy(path, bak)
        with open(path, "w", encoding="utf-8") as f: f.writelines(new_lines)
        print("  Patched: _negative_binomial.py")

def _patch_base_module(p):
    path = next((os.path.join(p, f) for f in ["module/base/_base_module.py",
                "scvi/module/base/_base_module.py"]
                 if os.path.exists(os.path.join(p, f))), None)
    if not path: return
    with open(path, "r", encoding="utf-8") as f: lines = f.readlines()
    _JAX = ("flax.", "jax.", "train_state.", "TrainState")
    targets = []
    for l in lines:
        m = re.match(r"\s*class\s+(\w+)\s*\(([^)]+)\)\s*:", l)
        if m and any(b.strip().startswith(px) for px in _JAX for b in [m.group(2)]):
            targets.append(m.group(1))
    if not targets: return
    for name in targets:
        start = next(i for i, l in enumerate(lines)
                     if re.match(rf"\s*class\s+{re.escape(name)}\b", l))
        indent = len(lines[start]) - len(lines[start].lstrip())
        end = start + 1
        while end < len(lines) and (not lines[end].strip()
               or len(lines[end]) - len(lines[end].lstrip()) > indent):
            end += 1
        ci = " " * indent
        lines = (lines[:start]
                 + [f"{ci}class {name}:  # [CPA] JAX stub\n", f"{ci}    pass\n"]
                 + lines[end:])
    bak = path + ".bak_cpa"
    if not os.path.exists(bak): shutil.copy(path, bak)
    with open(path, "w", encoding="utf-8") as f: f.writelines(lines)
    print(f"  Patched: _base_module.py — stubbed {len(targets)} JAX classes")

_scvi_path = _get_scvi_path()
if _scvi_path:
    _restore_backups(_scvi_path)
    _patch_types(_scvi_path)
    _patch_negative_binomial(_scvi_path)
    _patch_base_module(_scvi_path)
    print("  Pre-patching complete")
else:
    print("  scvi not found — skipping patches")

# =============================================================================
# STEP 5 — JAX/FLAX stub
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5/16: Installing JAX/FLAX runtime stub...")

class _JaxStub:
    def __init__(self, name="jax"):
        object.__setattr__(self, "__name__", name)
        object.__setattr__(self, "__file__", f"<stub:{name}>")
        object.__setattr__(self, "__path__", [])
        object.__setattr__(self, "__spec__", None)
        object.__setattr__(self, "_attrs", {})
    def __getattr__(self, attr):
        if attr.startswith("__"): raise AttributeError(attr)
        if attr in self._attrs: return self._attrs[attr]
        child = _JaxStub(f"{self.__name__}.{attr}")
        sys.modules[child.__name__] = child
        return child
    def __setattr__(self, attr, value):
        if attr in ("__name__","__file__","__path__","__spec__","_attrs"):
            object.__setattr__(self, attr, value)
        else:
            self._attrs[attr] = value
    def __call__(self, *a, **k): return self
    def __iter__(self): return iter([])

def _install_stub():
    mods = ["jax","jax.numpy","jax.random","jax.scipy","jax.nn","jax.tree_util",
            "jax.lib","jax.dlpack","flax","flax.linen","flax.training",
            "flax.serialization","flax.struct","flax.core",
            "flax.training.train_state","flax.optim","jaxlib","jaxlib.xla_extension"]
    for m in mods:
        sys.modules.setdefault(m, _JaxStub(m))
    class _Arr: __or__ = __ror__ = lambda s,o: o
    sys.modules["jax.numpy"].ndarray = _Arr()
    class _TS:
        __init__ = lambda s,*a,**k: None
        apply_gradients = lambda s,*a,**k: s
    sys.modules["flax.training.train_state"].TrainState = _TS
    sys.modules["flax.struct"].dataclass = lambda *a,**k: (lambda cls: cls)
    print("  JAX/FLAX stub installed")

_install_stub()

# =============================================================================
# STEP 6 — Patch CPA
# =============================================================================
print("\n" + "=" * 70)
print("STEP 6/16: Patching CPA...")

with open(os.path.join(CPA_DIR, "cpa", "__init__.py"), "w") as f:
    f.write('''import warnings; warnings.simplefilter("ignore")
from ._model import CPA
from ._module import CPAModule
from . import _plotting as pl
try: from ._api import ComPertAPI
except: ComPertAPI = None
try: from ._tuner import run_autotune
except: run_autotune = None
__version__ = "0.8.8"''')

_dp = os.path.join(CPA_DIR, "cpa", "_data.py")
with open(_dp) as f: dc = f.read()
if "from scvi.model._utils import parse_use_gpu_arg" in dc:
    dc = dc.replace("from scvi.model._utils import parse_use_gpu_arg",
                    """import torch as _ct
def parse_use_gpu_arg(use_gpu, return_device=False):
    a, d = ("gpu", _ct.device("cuda")) if (use_gpu and _ct.cuda.is_available()) else ("cpu", _ct.device("cpu"))
    return (a, None, d) if return_device else a""")
with open(_dp, "w") as f: f.write(dc)

_mp = os.path.join(CPA_DIR, "cpa", "_model.py")
with open(_mp) as f: mc = f.read()
mc = re.sub(r"from scvi\.train\._callbacks import SaveBestState", "# removed", mc)
mc = re.sub(r"checkpoint\s*=\s*SaveBestState\([^)]*\)", "# removed", mc)
mc = re.sub(r"callbacks\s*=\s*\[[^\]]*checkpoint[^\]]*\]", "callbacks = []", mc, flags=re.DOTALL)
with open(_mp, "w") as f: f.write(mc)

print("  CPA patches applied")

# =============================================================================
# STEP 7 — Import CPA + late imports
# =============================================================================
print("\n" + "=" * 70)
print("STEP 7/16: Importing CPA...")
sys.path.insert(0, CPA_DIR)
warnings.filterwarnings("ignore")
import cpa
print(f"  CPA {getattr(cpa,'__version__','?')} ready")

print("\n" + "=" * 70)
print("LATE IMPORTS: scanpy, scipy, sklearn...")
import scanpy as sc
import scipy.sparse as sp
from scipy.spatial.distance import cdist
from sklearn.metrics import r2_score
from tqdm import tqdm
print("  Late imports OK")

# =============================================================================
# STEP 8 — Model & data verification
# =============================================================================
print("\n" + "=" * 70)
print("STEP 8/16: Model & data verification...")
os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(PRETRAINED_PT):
    print("  Downloading Kang (K562) model.pt directly (35 MB)...")
    import gdown
    gdown.download(id=KANG_MODEL_ID, output=PRETRAINED_PT, quiet=False, use_cookies=False)
    print(f"  Downloaded → {PRETRAINED_PT}")
else:
    print(f"  Using existing model: {PRETRAINED_PT}")

# pandas shim + load checkpoint
import types
for pcm in ["pandas.core.indexes.numeric", "pandas.core.indexes.frozen", "pandas.core.indexes.range"]:
    if pcm not in sys.modules:
        stub = types.ModuleType(pcm)
        for pa in ("Int64Index","Float64Index","UInt64Index","Index","RangeIndex"):
            setattr(stub, pa, getattr(pd, pa, pd.Index))
        sys.modules[pcm] = stub

raw_ckpt = torch.load(PRETRAINED_PT, map_location="cpu", weights_only=False)
sd = raw_ckpt.get("model_state_dict") or raw_ckpt.get("state_dict") or raw_ckpt
ckpt_var_names = None

if isinstance(raw_ckpt, dict) and "var_names" in raw_ckpt:
    vn = raw_ckpt["var_names"]
    ckpt_var_names = (vn.tolist() if hasattr(vn, "tolist")
                      else list(vn) if isinstance(vn, (list, tuple)) else vn)

print(f"  Checkpoint genes: {len(ckpt_var_names)}")

attr = {}
pert_encoder = {}
if isinstance(raw_ckpt, dict) and "attr_dict" in raw_ckpt:
    ad_raw = raw_ckpt["attr_dict"]
    attr = dict(ad_raw.get("init_params_", {}).get("kwargs", {}).get("hyper_params", {}))
    registry = ad_raw.get("registry_", {})
    pert_encoder = registry.get("setup_args", {}).get("pert_encoder", {})

arch = {}
try:
    arch["n_genes"] = sd["px_r"].shape[0]
    arch["n_hidden"] = next(v for k, v in sd.items() if "fc_layers.Layer 0" in k and k.endswith("0.bias")).shape[0]
    arch["n_latent"] = sd["encoder.z.weight"].shape[0]
    arch["n_layers"] = len({m.group(1) for k in sd for m in [re.search(r"Layer (\d+)", k)] if m and "encoder" in k})
except Exception:
    pass

print(f"  Using model file: {PRETRAINED_PT}")

# =============================================================================
# STEP 9 — Gene name reconciliation
# =============================================================================
print("\n" + "=" * 70)
print("STEP 9/16: Gene name reconciliation...")
adata = sc.read_h5ad(DATA_PATH)
print(f"  Loaded: {adata.shape}")

_sample_h5ad = list(adata.var_names[:5])
_sample_ckpt = ckpt_var_names[:5] if ckpt_var_names else []
_h5ad_is_ensembl = any(str(g).startswith("ENSG") for g in _sample_h5ad)
_ckpt_is_symbol = (ckpt_var_names is not None and not any(str(g).startswith("ENSG") for g in _sample_ckpt))

print(f"  h5ad genes (sample):       {_sample_h5ad}")
print(f"  Checkpoint genes (sample):  {_sample_ckpt}")
print(f"  h5ad looks like Ensembl:    {_h5ad_is_ensembl}")
print(f"  Checkpoint looks like HGNC: {_ckpt_is_symbol}")

# --- Build Ensembl → Symbol mapping ---
_ensembl_to_symbol = {}
if ckpt_var_names:
    ckpt_set = set(ckpt_var_names)
    direct_overlap = set(adata.var_names) & ckpt_set

    if len(direct_overlap) >= 100:
        print(f"  ✓ Direct overlap: {len(direct_overlap)} — no conversion needed")
    else:
        print(f"  Direct overlap: {len(direct_overlap)} — conversion required")

        _symbol_col = None
        for col in adata.var.columns:
            vals = set(adata.var[col].astype(str).values)
            col_overlap = len(vals & ckpt_set)
            print(f"    Checking adata.var['{col}']: {col_overlap} matches")
            if col_overlap >= 100:
                _symbol_col = col
                break

        if _symbol_col:
            print(f"  ✓ Using adata.var['{_symbol_col}'] as gene symbol column")
            for idx, row in adata.var.iterrows():
                sym = str(row[_symbol_col]).strip()
                if sym and sym != "nan" and sym != "":
                    _ensembl_to_symbol[str(idx)] = sym
        elif _h5ad_is_ensembl and _ckpt_is_symbol:
            print("  Fetching Ensembl → HGNC mapping via pybiomart...")
            try:
                from pybiomart import Server
                server = Server(host="http://www.ensembl.org")
                dataset = server.marts["ENSEMBL_MART_ENSEMBL"].datasets["hsapiens_gene_ensembl"]
                raw_ids = [str(g) for g in adata.var_names]
                clean_ids = [re.sub(r'\.\d+$', '', g) for g in raw_ids]
                result = dataset.query(attributes=["ensembl_gene_id", "hgnc_symbol"],
                                       filters={"ensembl_gene_id": clean_ids})
                _clean_to_symbol = {}
                for _, row in result.iterrows():
                    eid = str(row["Gene stable ID"]).strip()
                    sym = str(row["HGNC symbol"]).strip()
                    if sym and sym != "nan" and sym != "" and eid:
                        _clean_to_symbol[eid] = sym
                for raw, clean in zip(raw_ids, clean_ids):
                    if clean in _clean_to_symbol:
                        _ensembl_to_symbol[raw] = _clean_to_symbol[clean]
                print(f"  ✓ pybiomart mapped {len(_ensembl_to_symbol)}/{len(raw_ids)} genes")
            except Exception as e:
                print(f"  ✗ pybiomart failed: {e}")
                print("    Trying MyGene.info fallback...")
                try:
                    import urllib.request, json as _json
                    raw_ids = [str(g) for g in adata.var_names]
                    clean_ids = [re.sub(r'\.\d+$', '', g) for g in raw_ids]
                    BATCH = 1000
                    _clean_to_symbol = {}
                    for i in range(0, len(clean_ids), BATCH):
                        batch = clean_ids[i:i+BATCH]
                        body = ("q=" + ",".join(batch) + "&scopes=ensembl.gene&fields=symbol&species=human")
                        req = urllib.request.Request("https://mygene.info/v3/query",
                                                     data=body.encode(),
                                                     headers={"Content-Type": "application/x-www-form-urlencoded"},
                                                     method="POST")
                        with urllib.request.urlopen(req, timeout=60) as resp:
                            results = _json.loads(resp.read().decode())
                            for hit in results:
                                if isinstance(hit, dict) and "symbol" in hit and "query" in hit:
                                    _clean_to_symbol[hit["query"]] = hit["symbol"]
                        print(f"    Batch {i//BATCH + 1}: {len(_clean_to_symbol)} symbols so far")
                    for raw, clean in zip(raw_ids, clean_ids):
                        if clean in _clean_to_symbol:
                            _ensembl_to_symbol[raw] = _clean_to_symbol[clean]
                    print(f"  ✓ MyGene mapped {len(_ensembl_to_symbol)}/{len(raw_ids)} genes")
                except Exception as e2:
                    print(f"  ✗ MyGene also failed: {e2}")

if _ensembl_to_symbol:
    converted_set = set(_ensembl_to_symbol.values())
    final_overlap = len(converted_set & ckpt_set)
    print(f"\nFINAL: {final_overlap} h5ad genes → checkpoint gene names")
else:
    print("\n⚠ No gene conversion was possible")

# =============================================================================
# STEP 10 — Data preparation
# =============================================================================
print("\n" + "=" * 70)
print("STEP 10/16: Preparing data...")

if ckpt_var_names:
    gene_list = ckpt_var_names
    X_src = adata.X.toarray() if sp.issparse(adata.X) else np.array(adata.X)
    X_src = X_src.astype(np.float32)

    if _ensembl_to_symbol:
        symbol_to_col = {}
        for i, g in enumerate(adata.var_names):
            g_str = str(g)
            sym = _ensembl_to_symbol.get(g_str, g_str)
            symbol_to_col[sym] = i
            symbol_to_col[g_str] = i
    else:
        symbol_to_col = {str(g): i for i, g in enumerate(adata.var_names)}

    X_full = np.zeros((adata.n_obs, len(gene_list)), dtype=np.float32)
    n_mapped = 0
    for col, ckpt_g in enumerate(gene_list):
        src_col = symbol_to_col.get(ckpt_g)
        if src_col is not None:
            X_full[:, col] = X_src[:, src_col]
            n_mapped += 1

    print(f"  Mapped {n_mapped}/{len(gene_list)} checkpoint genes to h5ad data")

    if n_mapped == 0:
        raise RuntimeError("ZERO genes mapped after conversion. Check STEP 9 output.")

    adata = sc.AnnData(X=sp.csr_matrix(X_full), obs=adata.obs.copy(), var=pd.DataFrame(index=gene_list))
    print(f"  Aligned to model genes: {adata.shape} ({n_mapped} non-zero)")

# --- Perturbation column ---
if "gene" in adata.obs.columns:
    adata.obs["perturbation"] = adata.obs["gene"].astype(str)
elif "perturbation" in adata.obs.columns:
    adata.obs["perturbation"] = adata.obs["perturbation"].astype(str)

adata.obs["perturbation"] = adata.obs["perturbation"].str.replace(CTRL_LABEL, "ctrl", regex=False)
CTRL_LABEL = "ctrl"

if pert_encoder:
    model_perts = set(pert_encoder.keys()) - {"<PAD>"}
    available = set(adata.obs["perturbation"].unique())
    keep_perts = model_perts & available
    keep_perts.add(CTRL_LABEL)
    if len(keep_perts) > 1:
        adata = adata[adata.obs["perturbation"].isin(keep_perts)].copy()
    else:
        print("  ⚠ pert_encoder overlap = 0, keeping all perturbations")

adata.obs["cell_type"] = "K562"
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()
adata.X = adata.layers["counts"].copy()

# --- Subsample ---
vc = adata.obs["perturbation"].value_counts()
adata = adata[adata.obs["perturbation"].isin(vc[vc >= 5].index)].copy()
rng = np.random.default_rng(SEED)
keep = []
for p, g in adata.obs.groupby("perturbation"):
    if p == CTRL_LABEL:
        n = max(1, int(len(g) * max(SUBSAMPLE, 0.30)))
    else:
        n = max(1, int(len(g) * SUBSAMPLE))
    keep.extend(rng.choice(g.index.tolist(), n, replace=False))
adata = adata[keep].copy()

vc2 = adata.obs["perturbation"].value_counts()
_drop = vc2[(vc2 < 2) & (vc2.index != CTRL_LABEL)].index
adata = adata[~adata.obs["perturbation"].isin(_drop)].copy()

print(f"  After subsample + filter: {adata.shape}")
print(f"  Control cells: {int((adata.obs['perturbation'] == CTRL_LABEL).sum())}")

# =============================================================================
# STEP 11 — DEGs and splits
# =============================================================================
print("\n" + "=" * 70)
print("STEP 11/16: Computing DEGs and splits...")

adata.obs["cov_cond"] = "K562_" + adata.obs["perturbation"].astype(str)

if sp.issparse(adata.X):
    _mean = np.asarray(adata.X.mean(axis=0)).flatten()
    _sq_mean = np.asarray(adata.X.power(2).mean(axis=0)).flatten()
    real_var = _sq_mean - _mean ** 2
else:
    real_var = np.asarray(adata.X.var(axis=0)).flatten()

_real_genes_mask = real_var > 1e-8
_real_gene_names = adata.var_names[_real_genes_mask]
n_real = int(_real_genes_mask.sum())
print(f"  Non-zero-variance genes: {n_real}/{adata.n_vars}")

if n_real < 10:
    raise RuntimeError(f"Only {n_real} genes have non-zero variance — too few for DEG.")

_ctrl_n = int((adata.obs["perturbation"] == CTRL_LABEL).sum())
print(f"  Control group '{CTRL_LABEL}': {_ctrl_n} cells")

if _ctrl_n < 2:
    raise RuntimeError(f"Control group has {_ctrl_n} cells — need ≥ 2 for DEG reference.")

_grp_counts = adata.obs["perturbation"].value_counts()
_small_grps = [g for g in _grp_counts[_grp_counts < 3].index if g != CTRL_LABEL]
if _small_grps:
    print(f"  Dropping {len(_small_grps)} groups with < 3 cells")
    adata = adata[~adata.obs["perturbation"].isin(_small_grps)].copy()

_adata_rgg = adata[:, _real_gene_names].copy()
sc.pp.normalize_total(_adata_rgg, target_sum=1e4)
sc.pp.log1p(_adata_rgg)

_N_GENES_DEG = min(200, len(_real_gene_names))
_deg_ok = False
for _deg_method in ["t-test", "wilcoxon"]:
    try:
        print(f"  Trying rank_genes_groups method='{_deg_method}' (n_genes={_N_GENES_DEG})...")
        sc.tl.rank_genes_groups(_adata_rgg, groupby="perturbation", reference=CTRL_LABEL,
                                method=_deg_method, n_genes=_N_GENES_DEG,
                                key_added="rank_genes_groups", use_raw=False)
        _rgg = _adata_rgg.uns.get("rank_genes_groups")
        if _rgg is not None and "names" in _rgg and _rgg["names"] is not None and len(_rgg["names"]) > 0:
            _deg_ok = True
            print(f"  ✓ DEG succeeded with '{_deg_method}'")
            break
        else:
            print(f"  ✗ '{_deg_method}' returned empty results")
    except Exception as _e:
        print(f"  ✗ '{_deg_method}' failed: {type(_e).__name__}: {_e}")

if not _deg_ok:
    raise RuntimeError("All DEG methods failed. Check data integrity.")

adata.uns["rank_genes_groups"] = _adata_rgg.uns["rank_genes_groups"]
del _adata_rgg

adata.uns["rank_genes_groups_cov"] = {
    f"K562_{g}": list(adata.uns["rank_genes_groups"]["names"][g])
    for g in adata.uns["rank_genes_groups"]["names"].dtype.names if g != CTRL_LABEL
}

_all_perts = [p for p in adata.obs["perturbation"].unique() if p != CTRL_LABEL]
_ood = (rng.choice(_all_perts, max(1, int(len(_all_perts) * 0.1)), replace=False).tolist() if _all_perts else [])
adata.obs["split"] = rng.choice(["train", "valid"], adata.n_obs, p=[0.85, 0.15])
if _ood:
    adata.obs.loc[adata.obs["perturbation"].isin(_ood), "split"] = "ood"

# =============================================================================
# STEP 12 — Setup + build model (CRITICAL FIXES)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 12/16: Setting up CPA model...")

cpa.CPA.setup_anndata(adata, perturbation_key="perturbation", control_group=CTRL_LABEL,
                      categorical_covariate_keys=["cell_type"], is_count_data=True,
                      deg_uns_key="rank_genes_groups_cov", deg_uns_cat_key="cov_cond", max_comb_len=1)

# --- Extract architecture from checkpoint weight shapes ---
_enc_hidden, _dec_hidden = None, None
_enc_layers, _dec_layers = 0, 0
for k, v in sd.items():
    if "encoder" in k and "fc_layers.Layer" in k and k.endswith("0.bias"):
        _enc_hidden = v.shape[0]
        _layer_n = int(re.search(r"Layer (\d+)", k).group(1))
        _enc_layers = max(_enc_layers, _layer_n + 1)
    if "decoder" in k and "fc_layers.Layer" in k and k.endswith("0.bias"):
        _dec_hidden = v.shape[0]
        _layer_n = int(re.search(r"Layer (\d+)", k).group(1))
        _dec_layers = max(_dec_layers, _layer_n + 1)

print(f"  Checkpoint architecture:")
print(f"    encoder: hidden={_enc_hidden}, layers={_enc_layers}")
print(f"    decoder: hidden={_dec_hidden}, layers={_dec_layers}")
print(f"    n_latent={arch.get('n_latent', '?')}")

# --- Build kwargs from checkpoint attr_dict + architecture ---
cpa_kw = {}
for k in ("n_latent", "n_layers_encoder", "n_hidden_encoder", "dropout_rate",
          "use_batch_norm", "n_hidden_decoder", "n_layers_decoder"):
    if k in attr:
        cpa_kw[k] = attr[k]

if "n_latent" not in cpa_kw and "n_latent" in arch:
    cpa_kw["n_latent"] = arch["n_latent"]
if _enc_hidden and "n_hidden_encoder" not in cpa_kw:
    cpa_kw["n_hidden_encoder"] = _enc_hidden
if _enc_layers and "n_layers_encoder" not in cpa_kw:
    cpa_kw["n_layers_encoder"] = _enc_layers
if _dec_hidden and "n_hidden_decoder" not in cpa_kw:
    cpa_kw["n_hidden_decoder"] = _dec_hidden
if _dec_layers and "n_layers_decoder" not in cpa_kw:
    cpa_kw["n_layers_decoder"] = _dec_layers

# --- Introspect CPAModule to find accepted parameters ---
import inspect
_mod_sig = set(inspect.signature(cpa.CPAModule.__init__).parameters.keys()) - {"self"}
_cpa_has_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD
                      for p in inspect.signature(cpa.CPA.__init__).parameters.values())
print(f"  CPAModule accepted params: {sorted(_mod_sig)}")

# --- CRITICAL: Remove n_hidden (never accepted by CPAModule) ---
if "n_hidden" in cpa_kw:
    if "n_hidden_encoder" not in cpa_kw:
        cpa_kw["n_hidden_encoder"] = cpa_kw["n_hidden"]
    cpa_kw.pop("n_hidden")
    print(f"  Removed n_hidden from kwargs")

# --- Remove anything CPAModule won't accept ---
if not _cpa_has_kwargs:
    _rejected = {k for k in cpa_kw if k not in _mod_sig}
    if _rejected:
        print(f"  Removing unsupported kwargs: {_rejected}")
        for k in _rejected:
            cpa_kw.pop(k)

print(f"  Final CPA kwargs: {cpa_kw}")

# --- Initialize model ---
model = cpa.CPA(adata=adata, **cpa_kw)

# --- Shape-filtered state_dict loading ---
model_sd = model.module.state_dict()
filtered_sd = {}
skipped_keys = []
for k, v in sd.items():
    if k in model_sd:
        if model_sd[k].shape == v.shape:
            filtered_sd[k] = v
        else:
            skipped_keys.append(f"    {k}: ckpt {tuple(v.shape)} → model {tuple(model_sd[k].shape)}")

_missing = set(model_sd.keys()) - set(sd.keys())
model.module.load_state_dict(filtered_sd, strict=False)
n_loaded = len(filtered_sd)
n_total = len(model_sd)
print(f"  Loaded {n_loaded}/{n_total} parameter tensors from checkpoint")

if skipped_keys:
    print(f"  Skipped {len(skipped_keys)} size-mismatched tensors (expected — different #perts/#covars):")
    for s in skipped_keys[:8]:
        print(s)
    if len(skipped_keys) > 8:
        print(f"    ... and {len(skipped_keys) - 8} more")

if _missing:
    print(f"  {len(_missing)} model params not in checkpoint (random init)")

# --- Move to device (FIXED: explicit device argument) ---
_device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    model.to_device(_device)
except TypeError:
    model.module.to(torch.device(_device))
print(f"  Model ready on {_device} (n_genes={adata.n_vars})")

# =============================================================================
# STEP 13 — Predictions (CRITICAL FIXES)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 13/16: Running CPA predictions...")

adata.layers["X_true"] = adata.X.copy()

_ctrl_mask = adata.obs["perturbation"] == CTRL_LABEL
_ctrl_sub = adata[_ctrl_mask].copy()

if _ctrl_sub.n_obs == 0:
    print("  ⚠ No ctrl cells — using dataset mean as input baseline")
    _cX = adata.layers["counts"]
    _cX = _cX.toarray() if sp.issparse(_cX) else np.array(_cX)
    _samp = np.tile(_cX.mean(0, keepdims=True), (adata.n_obs, 1))
else:
    _cX = _ctrl_sub.X.toarray() if sp.issparse(_ctrl_sub.X) else np.array(_ctrl_sub.X)
    _samp = _cX[rng.choice(_ctrl_sub.n_obs, size=adata.n_obs, replace=True)]

_s32 = _samp.astype(np.float32)
adata.X = sp.csr_matrix(_s32) if sp.issparse(adata.layers["X_true"]) else _s32

# --- Ensure model on device ---
try:
    model.to_device(_device)
except TypeError:
    model.module.to(torch.device(_device))

model.predict(adata, batch_size=512)

# --- Robust prediction key search ---
_pred_found = False
for _key_loc, _key_dict in [("obsm", adata.obsm), ("layers", adata.layers)]:
    for _k in list(_key_dict.keys()):
        if "pred" in _k.lower() or _k == "CPA_pred":
            adata.layers["CPA_pred"] = np.array(_key_dict[_k])
            _pred_found = True
            print(f"  Found predictions in adata.{_key_loc}['{_k}']")
            break
    if _pred_found:
        break

if not _pred_found:
    raise KeyError("CPA predictions not found in adata.obsm or adata.layers.\n"
                   f"  Available obsm keys: {list(adata.obsm.keys())}\n"
                   f"  Available layer keys: {list(adata.layers.keys())}")

adata.X = adata.layers["X_true"].copy()
print(f"  Predictions ready: {adata.layers['CPA_pred'].shape}")
# =============================================================================
# STEP 14 — R² metrics (FIXED: log1p-normalized space)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 14/16: Computing R² metrics...")

# --- Normalize predictions and true counts to log1p space ---
print("  Normalizing predictions and true counts to log1p space...")

# Get non-zero-variance genes for metric computation
if sp.issparse(adata.layers["counts"]):
    _mean    = np.asarray(adata.layers["counts"].mean(axis=0)).flatten()
    _sq_mean = np.asarray(adata.layers["counts"].power(2).mean(axis=0)).flatten()
    real_var = _sq_mean - _mean ** 2
else:
    real_var = np.asarray(adata.layers["counts"].var(axis=0)).flatten()

_real_genes_mask = real_var > 1e-8
_real_gene_idx = np.where(_real_genes_mask)[0]
n_real = int(_real_genes_mask.sum())
print(f"  Using {n_real}/{adata.n_vars} non-zero-variance genes for metrics")

# Library size normalization + log1p for TRUE counts
_true_raw = adata.layers["counts"]
_lib_size = np.asarray(_true_raw.sum(axis=1)).flatten()
_lib_size = np.maximum(_lib_size, 1.0)  # Avoid division by zero
_true_norm = (_true_raw / _lib_size[:, None]) * 1e4
if sp.issparse(_true_norm):
    _true_norm = _true_norm.toarray()
_true_log = np.log1p(_true_norm).astype(np.float64)

# Same normalization for PREDICTIONS
_pred_raw = adata.layers["CPA_pred"]
_lib_size_pred = np.asarray(_pred_raw.sum(axis=1)).flatten()
_lib_size_pred = np.maximum(_lib_size_pred, 1.0)
_pred_norm = (_pred_raw / _lib_size_pred[:, None]) * 1e4
if not isinstance(_pred_norm, np.ndarray):
    _pred_norm = np.asarray(_pred_norm)
_pred_log = np.log1p(_pred_norm).astype(np.float64)

# Store normalized versions for metric computation
adata.layers["counts_log"] = _true_log
adata.layers["CPA_pred_log"] = _pred_log

# Control baseline in log space
_ctrl_for_mu = adata[adata.obs["perturbation"] == CTRL_LABEL].copy()
if _ctrl_for_mu.n_obs == 0:
    print("  ⚠ No ctrl cells for baseline — using global mean")
    ctrl_mu = _true_log.mean(0)
else:
    ctrl_mu = _ctrl_for_mu.layers["counts_log"].mean(0)

def _safe_r2(y_true, y_pred):
    """r2_score that handles edge cases gracefully."""
    if len(y_true) < 2:
        return np.nan
    try:
        return float(r2_score(y_true, y_pred))
    except ValueError:
        return np.nan

r2_res = defaultdict(list)
for _cond in tqdm(adata.obs["perturbation"].unique(), desc="R²", ncols=70):
    if _cond == CTRL_LABEL:
        continue
    _dk = f"K562_{_cond}"
    if _dk not in adata.uns.get("rank_genes_groups_cov", {}):
        continue
    _m  = adata.obs["perturbation"] == _cond
    _xt = adata[_m].layers["counts_log"]  # LOG-SPACE
    _xp = adata[_m].layers["CPA_pred_log"]  # LOG-SPACE
    _dg = adata.uns["rank_genes_groups_cov"][_dk]

    for _nt in [10, 20, 50, None]:
        _lb = _nt if _nt is not None else "all"
        if _nt is not None:
            _idx = np.where(np.isin(adata.var_names, _dg[:_nt]))[0]
            # Restrict to real genes only
            _idx = np.intersect1d(_idx, _real_gene_idx)
        else:
            _idx = _real_gene_idx  # Use only non-zero-variance genes

        if len(_idx) == 0:
            continue

        _mt = _xt[:, _idx].mean(0)
        _mp = _xp[:, _idx].mean(0)
        _mc = ctrl_mu[_idx]

        r2_res["condition"].append(_cond)
        r2_res["n_top_deg"].append(_lb)
        r2_res["r2_mean_deg"].append(_safe_r2(_mt, _mp))
        r2_res["r2_mean_lfc_deg"].append(_safe_r2(_mt - _mc, _mp - _mc))

df_r2 = pd.DataFrame(r2_res)
if len(df_r2) > 0:
    print("\nR² by top-N DEGs (log1p-normalized space):")
    print(df_r2.groupby("n_top_deg")[["r2_mean_deg", "r2_mean_lfc_deg"]].mean().to_string())
else:
    print("\n⚠ No valid conditions for R² computation")

# =============================================================================
# STEP 15 — 3-tier metrics (FIXED: log1p-normalized space)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 15/16: Computing 3-tier metrics...")

# --- LIMIT EVALUATION SET for memory efficiency ---
MAX_EVAL_PERTS = 100
MAX_CELLS_SAMPLE = 150

_eval = [c for c in adata.obs["perturbation"].unique()
         if c != CTRL_LABEL
         and f"K562_{c}" in adata.uns.get("rank_genes_groups_cov", {})]

if len(_eval) > MAX_EVAL_PERTS:
    _eval_counts = [(c, (adata.obs["perturbation"] == c).sum()) for c in _eval]
    _eval_counts.sort(key=lambda x: x[1], reverse=True)
    _eval = [c for c, _ in _eval_counts[:MAX_EVAL_PERTS]]
    print(f"  Limited evaluation to top {MAX_EVAL_PERTS} perturbations by cell count")

if len(_eval) == 0:
    print("  ⚠ No evaluable perturbations found — skipping 3-tier metrics")
    df = pd.DataFrame()
    P = 0
else:
    def _pearson(a, b):
        if a.std() < 1e-8 or b.std() < 1e-8:
            return np.nan
        r = np.corrcoef(a, b)[0, 1]
        return np.nan if np.isnan(r) else float(r)

    def _da(pm, tm, cm, thr=0.1):
        dp, dt = pm - cm, tm - cm
        m = np.abs(dt) > thr
        if not m.any():
            return np.nan
        return float(np.mean(np.sign(dp[m]) == np.sign(dt[m])))

    def _jaccard(pm, tm, cm, k=50):
        pt = set(np.argsort(np.abs(pm - cm))[-k:].tolist())
        tt = set(np.argsort(np.abs(tm - cm))[-k:].tolist())
        u = pt | tt
        return len(pt & tt) / len(u) if u else 0.0

    def _energy(p, q, n=MAX_CELLS_SAMPLE):
        _rng = np.random.default_rng(0)
        if len(p) < 2 or len(q) < 2:
            return np.nan
        n_p = min(n, len(p))
        n_q = min(n, len(q))
        p = p[_rng.choice(len(p), n_p, replace=False)].astype(np.float32)
        q = q[_rng.choice(len(q), n_q, replace=False)].astype(np.float32)
        return max(float(2 * cdist(p, q).mean()
                        - cdist(p, p).mean()
                        - cdist(q, q).mean()), 0.)

    def _mmd(p, q, n=MAX_CELLS_SAMPLE):
        _rng = np.random.default_rng(0)
        if len(p) < 2 or len(q) < 2:
            return np.nan
        n_p = min(n, len(p))
        n_q = min(n, len(q))
        p = p[_rng.choice(len(p), n_p, replace=False)].astype(np.float32)
        q = q[_rng.choice(len(q), n_q, replace=False)].astype(np.float32)
        dqq = cdist(q, q, "sqeuclidean")
        off = dqq[~np.eye(len(q), dtype=bool)]
        s2  = max(float(np.median(off)) / 2.0 if len(off) else 1.0, 1e-6)
        Kpp = np.exp(-cdist(p, p, "sqeuclidean") / (2 * s2)).mean()
        Kqq = np.exp(-dqq / (2 * s2)).mean()
        Kpq = np.exp(-cdist(p, q, "sqeuclidean") / (2 * s2)).mean()
        return max(float(Kpp - 2 * Kpq + Kqq), 0.)

    # --- Collect centroids (LOG-SPACE, MEMORY-SAFE BATCHES) ---
    PC, TC, CN = [], [], []
    BATCH_SIZE = 50

    print(f"  Computing centroids for {len(_eval)} perturbations (log1p space)...")
    for i in range(0, len(_eval), BATCH_SIZE):
        batch = _eval[i:i+BATCH_SIZE]
        for _c in batch:
            _m  = adata.obs["perturbation"] == _c
            _xt = adata[_m].layers["counts_log"][:, _real_gene_idx]  # LOG + REAL GENES
            _xp = adata[_m].layers["CPA_pred_log"][:, _real_gene_idx]
            PC.append(_xp.mean(0))
            TC.append(_xt.mean(0))
            CN.append(_c)
        del _xt, _xp
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"    Batch {i//BATCH_SIZE + 1}/{(len(_eval)-1)//BATCH_SIZE + 1} complete")

    P  = len(CN)
    PM = np.stack(PC)
    TM = np.stack(TC)
    del PC, TC

    # --- Centroid Accuracy & PDS ---
    print(f"  Computing distance matrix for {P} perturbations...")
    D = np.linalg.norm(PM[:, None] - TM[None], axis=-1)
    ca_v = (D.argmin(1) == np.arange(P)).astype(float)
    d_self = D[np.arange(P), np.arange(P)]

    if P > 1:
        _off    = ~np.eye(P, dtype=bool)
        d_cross = (D * _off).sum(1) / _off.sum(1)
        pds_v   = d_self / (d_self + d_cross + 1e-8)
    else:
        pds_v = np.array([np.nan])

    del D, PM, TM
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --- Per-perturbation metrics ---
    res = defaultdict(list)
    for i, _c in enumerate(tqdm(CN, ncols=70, desc="3-tier")):
        _m  = adata.obs["perturbation"] == _c
        _xt = adata[_m].layers["counts_log"][:, _real_gene_idx]
        _xp = adata[_m].layers["CPA_pred_log"][:, _real_gene_idx]
        _dk = f"K562_{_c}"
        _deg_list = adata.uns["rank_genes_groups_cov"].get(_dk, [])
        _de = np.where(np.isin(adata.var_names[_real_gene_idx], _deg_list[:TOP_K_DE]))[0]

        pm = _xp.mean(0)
        tm = _xt.mean(0)

        res["condition"].append(_c)
        res["T1_CA"].append(ca_v[i])
        res["T1_PDS"].append(pds_v[i])
        res["T1_PR"].append(_pearson(pm - ctrl_mu[_real_gene_idx], tm - ctrl_mu[_real_gene_idx]))
        res["T2_DA"].append(_da(pm, tm, ctrl_mu[_real_gene_idx]))

        if len(_de) > 0:
            res["T2_PRde"].append(
                _pearson(pm[_de] - ctrl_mu[_real_gene_idx][_de],
                        tm[_de] - ctrl_mu[_real_gene_idx][_de]))
        else:
            res["T2_PRde"].append(np.nan)

        res["T2_JC"].append(_jaccard(pm, tm, ctrl_mu[_real_gene_idx], TOP_K_DE))
        res["T3_EN"].append(_energy(_xp, _xt))
        res["T3_MM"].append(_mmd(_xp, _xt))

        del _xt, _xp, pm, tm
        if i % 20 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    df = pd.DataFrame(res)
    print(f"  Completed metrics for {len(df)} perturbations")

# =============================================================================
# STEP 16 — FINAL RESULTS
# =============================================================================
SEP = "=" * 66
print(f"\n{SEP}")
print("  CPA (pretrained) · K562 · Final Evaluation")
print("  ✓ All metrics computed in log1p(library-size-normalized) space")
print("  ✓ Restricted to non-zero-variance genes only")

if P == 0:
    print("  ⚠ No evaluable perturbations — cannot report metrics.")
    print(SEP)
else:
    if _ensembl_to_symbol and ckpt_var_names:
        _conv_overlap = len(set(_ensembl_to_symbol.values()) & set(ckpt_var_names))
        print(f"  Gene conversion: {_conv_overlap}/{len(ckpt_var_names)} checkpoint genes matched")

    print(f"  {P} perturbations | {int(SUBSAMPLE*100)}% subsample | top-{TOP_K_DE} DEGs")
    print(f"  {'Tier  Metric':<44} {'Value':>8}  Dir")
    print(f"  {'-'*60}")

    def _row(tag, name, col, hi):
        if col not in df.columns:
            v = np.nan
        else:
            v = df[col].mean()
        s = f"{v:.4f}" if not np.isnan(v) else "   N/A"
        print(f"  [{tag}] {name:<42} {s:>8}  {'↑' if hi else '↓'}")

    _row("T1", "Centroid Accuracy (CA)",           "T1_CA",   True)
    _row("T1", "Profile Distance Score (PDS)",     "T1_PDS",  False)
    _row("T1", "Systema Pearson Delta",            "T1_PR",   True)
    _row("T2", "Directional Accuracy",             "T2_DA",   True)
    _row("T2", f"Pearson Delta DE Top-{TOP_K_DE}", "T2_PRde", True)
    _row("T2", f"Jaccard DE Top-{TOP_K_DE}",       "T2_JC",   True)
    _row("T3", "Energy Distance",                  "T3_EN",   False)
    _row("T3", "MMD (RBF kernel)",                 "T3_MM",   False)

    print(SEP)
    print("  Evaluation complete. ✅")

STEP 1/16: Cloning CPA repository...
  Cloned → /tmp/theislab_cpa

STEP 2/16: Installing dependencies...
  NumPy: 2.0.2 | PyTorch: 2.10.0+cu128
  All dependencies installed successfully

STEP 3/16: Clearing module cache...
  Cleared 1312 modules

STEP 4/16: Pre-patching scvi-tools...
  Patched: _types.py
  Patched: _base_module.py — stubbed 2 JAX classes
  Pre-patching complete

STEP 5/16: Installing JAX/FLAX runtime stub...
  JAX/FLAX stub installed

STEP 6/16: Patching CPA...
  CPA patches applied

STEP 7/16: Importing CPA...
  CPA 0.8.8 ready

LATE IMPORTS: scanpy, scipy, sklearn...
  Late imports OK

STEP 8/16: Model & data verification...
  Using existing model: ./pretrained_cpa_k562/k562_model.pt
  Checkpoint genes: 5000
  Using model file: ./pretrained_cpa_k562/k562_model.pt

STEP 9/16: Gene name reconciliation...
  Loaded: (310385, 8563)
  h5ad genes (sample):       ['ENSG00000237491', 'ENSG00000228794', 'ENSG00000188976', 'ENSG00000187961', 'ENSG00000188290']
  Checkpoint gene

100%|██████████| 2037/2037 [00:02<00:00, 860.27it/s]


INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        


INFO: Seed set to 0
INFO: Seed set to 0
INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0


  Checkpoint architecture:
    encoder: hidden=128, layers=2
    decoder: hidden=512, layers=2
    n_latent=64
  CPAModule accepted params: ['covars_encoder', 'doser_type', 'dropout_rate_decoder', 'dropout_rate_encoder', 'drug_embeddings', 'n_genes', 'n_hidden_decoder', 'n_hidden_doser', 'n_hidden_encoder', 'n_latent', 'n_layers_decoder', 'n_layers_doser', 'n_layers_encoder', 'n_perts', 'recon_loss', 'seed', 'use_batch_norm_decoder', 'use_batch_norm_encoder', 'use_layer_norm_decoder', 'use_layer_norm_encoder', 'variational']
  Final CPA kwargs: {'n_latent': 64, 'n_layers_encoder': 2, 'n_hidden_encoder': 128, 'n_hidden_decoder': 512, 'n_layers_decoder': 2}
  Loaded 27/41 parameter tensors from checkpoint
  Skipped 4 size-mismatched tensors (expected — different #perts/#covars):
    pert_network.pert_embedding.weight: ckpt (3, 64) → model (2038, 64)
    pert_network.dosers.beta: ckpt (1, 3) → model (1, 2038)
    pert_network.dosers.bias: ckpt (1, 3) → model (1, 2038)
    covars_embedding

...

  Found predictions in adata.obsm['CPA_pred']
  Predictions ready: (62279, 5000)

STEP 14/16: Computing R² metrics...
  Normalizing predictions and true counts to log1p space...
  Using 2602/5000 non-zero-variance genes for metrics


R²: 100%|█████████████████████████| 2037/2037 [00:56<00:00, 36.29it/s]



R² by top-N DEGs (log1p-normalized space):
           r2_mean_deg  r2_mean_lfc_deg
n_top_deg                              
10          -12.387979      -247.403709
20           -7.329057      -213.997019
50           -4.441489      -202.378619
all          -0.651448       -25.065478

STEP 15/16: Computing 3-tier metrics...
  Limited evaluation to top 100 perturbations by cell count
  Computing centroids for 100 perturbations (log1p space)...
    Batch 1/2 complete
    Batch 2/2 complete
  Computing distance matrix for 100 perturbations...


3-tier: 100%|███████████████████████| 100/100 [00:09<00:00, 10.54it/s]

  Completed metrics for 100 perturbations

  CPA (pretrained) · K562 · Final Evaluation
  ✓ All metrics computed in log1p(library-size-normalized) space
  ✓ Restricted to non-zero-variance genes only
  Gene conversion: 2602/5000 checkpoint genes matched
  100 perturbations | 20% subsample | top-50 DEGs
  Tier  Metric                                    Value  Dir
  ------------------------------------------------------------
  [T1] Centroid Accuracy (CA)                       0.0100  ↑
  [T1] Profile Distance Score (PDS)                 0.4996  ↓
  [T1] Systema Pearson Delta                        0.2734  ↑
  [T2] Directional Accuracy                         0.6010  ↑
  [T2] Pearson Delta DE Top-50                      0.0519  ↑
  [T2] Jaccard DE Top-50                            0.0592  ↑
  [T3] Energy Distance                             51.7200  ↓
  [T3] MMD (RBF kernel)                             0.6072  ↓
  Evaluation complete. ✅
